1. The basic setup of the experiment

In [ ]:
# ================================
# Block 1: Definitions and data generation (unchanged, additionally save common data)
# ================================
from __future__ import annotations
import json, math, os, random, time, warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import minimize
from scipy.sparse import csc_matrix, lil_matrix
from scipy.sparse.linalg import factorized
warnings.filterwarnings("ignore", category=RuntimeWarning)

# -------------------------------
# Reproducibility and configuration
# -------------------------------
def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

@dataclass
class Config:
    # ======== GENERAL SETUP ========
    seed: int = 7                     # Random seed for reproducibility
    fast_mode: bool = True            # Fast mode: uses few HYCO rounds and small networks for quick debugging
    run_hyco: bool = True             # Enable HYCO (physical + neural coupling) training
    run_pinn: bool = True             # Enable standard PINN training
    run_xpinn: bool = True            # Enable XPINN (domain-decomposed) training
    output_root: str = "Experiment3_outputs_navier_stokes"  # Root directory for all outputs
    dpi: int = 600                    # DPI for saved figures

    # ======== GEOMETRY & PHYSICS ========
    r_inner: float = 0.25             # Inner radius of the annular domain
    r_outer: float = 1.00             # Outer radius of the annular domain
    final_time: float = 1.50          # Total simulation time (T_end)
    nu_true: float = 1.0e-3           # True kinematic viscosity (ground truth)
    rho_true: float = 1.0             # True forcing amplitude (ground truth)
    noise_level: float = 0.00         # Relative noise std added to observations (0 = no noise)

    # ======== PARAMETER BOUNDS (for inversion) ========
    nu_min: float = 2.0e-4            # Lower bound for viscosity estimation
    nu_max: float = 5.0e-3            # Upper bound for viscosity estimation
    rho_min: float = 0.25             # Lower bound for forcing amplitude estimation
    rho_max: float = 1.75             # Upper bound for forcing amplitude estimation

    # ======== HYCO LOSS WEIGHTS ========
    w_data_phy: float = 2.0           # Weight for physical sector data loss
    w_data_syn: float = 4.0           # Weight for synthetic sector data loss
    w_interaction: float = 2.0        # Weight for ghost-point interaction loss
    w_ic_syn: float = 2.0             # Weight for initial condition loss for synthetic net
    w_bc_syn: float = 0.5             # Weight for boundary condition loss for synthetic net

    # ======== PINN / XPINN LOSS WEIGHTS ========
    w_pinn_data: float = 10.0         # Data-fitting loss weight for PINN/XPINN
    w_pinn_pde: float = 1.0           # PDE residual weight (vorticity transport)
    w_pinn_poisson: float = 1.0       # Poisson equation residual weight (ω + ∇²ψ = 0)
    w_pinn_ic: float = 3.0            # Initial condition loss weight
    w_pinn_bc: float = 1.0            # Boundary condition loss weight
    w_xpinn_interface: float = 2.0    # Interface continuity loss weight for XPINN

    def finalize(self) -> "Config":
        """Set hyperparameters based on fast_mode flag."""
        if self.fast_mode:
            # --- Grid & time stepping (coarse/fast) ---
            self.nr_true = 128                # Radial points for ground-truth solver
            self.ntheta_true = 256            # Angular points for ground-truth solver
            self.nr_phy = 64                  # Radial points for HYCO physical proxy
            self.ntheta_phy = 128             # Angular points for HYCO physical proxy
            self.dt_true = 3.5e-3             # Time step for ground-truth solver
            self.dt_phy = 5.0e-3              # Time step for HYCO physical proxy

            # --- Dataset sizes ---
            self.n_store = 13                 # Number of stored temporal snapshots
            self.n_obs_per_agent = 2500       # Observations per sector-agent
            self.n_validation = 3000         # Validation set size
            self.n_ghost = 3000               # Ghost points per HYCO round

            # --- HYCO optimization ---
            self.hyco_rounds = 150            # HYCO federated rounds (low for fast debug)
            self.hyco_syn_steps = 100         # Adam steps for synthetic network per round
            self.hyco_phys_maxiter = 20       # L-BFGS iterations for physical solver per round

            # --- Neural network architecture ---
            self.nn_width = 80                # Neurons per hidden layer
            self.nn_depth = 4                 # Number of hidden layers

            # --- Training epochs ---
            self.pinn_epochs = 10000            # PINN training epochs (fast)
            self.xpinn_epochs = 10000           # XPINN training epochs (fast)

            # --- Collocation & boundary sampling ---
            self.n_collocation = 512        # PDE collocation points
            self.n_ic = 2000                   # Initial condition points
            self.n_bc = 2000                   # Boundary condition points
            self.n_interface = 2000          # Total interface points for XPINN

            # --- Batch processing ---
            self.batch_data = 512             # Mini-batch size for data loss
            self.eval_batch = 8192            # Batch size for inference/validation
        else:
            # --- Full (standard) mode: higher accuracy, slower ---
            self.nr_true = 96                 # Radial points for ground-truth solver
            self.ntheta_true = 192            # Angular points for ground-truth solver
            self.nr_phy = 54                  # Radial points for HYCO physical proxy
            self.ntheta_phy = 96              # Angular points for HYCO physical proxy
            self.dt_true = 3.5e-3             # Time step for ground-truth solver
            self.dt_phy = 6.0e-3              # Time step for HYCO physical proxy
            self.n_store = 25                 # Number of stored temporal snapshots
            self.n_obs_per_agent = 1500       # Observations per sector-agent
            self.n_validation = 2200          # Validation set size
            self.n_ghost = 2200               # Ghost points per HYCO round
            self.hyco_rounds = 1000           # HYCO rounds for full convergence
            self.hyco_syn_steps = 180         # Adam steps for synthetic network per round
            self.hyco_phys_maxiter = 6        # L-BFGS iterations for physical solver
            self.nn_width = 128               # Neurons per hidden layer
            self.nn_depth = 5                 # Number of hidden layers
            self.pinn_epochs = 6000           # PINN training epochs (full)
            self.xpinn_epochs = 8000          # XPINN training epochs (full)
            self.n_collocation = 5000         # PDE collocation points
            self.n_ic = 1800                  # Initial condition points
            self.n_bc = 1600                  # Boundary condition points
            self.n_interface = 1200           # Total interface points for XPINN
            self.batch_data = 1024            # Mini-batch size for data loss
            self.eval_batch = 16384           # Batch size for inference/validation
        return self

CFG = Config().finalize()
set_seed(CFG.seed)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
DTYPE = torch.float32

ROOT = Path(CFG.output_root)
for sub in ["figures", "tables", "data", "checkpoints", "logs"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
with open(ROOT / "configuration.json", "w") as f:
    json.dump(asdict(CFG), f, indent=2)
print(f"Device: {DEVICE}, Output root: {ROOT.resolve()}")

# -------------------------------
# Geometry and forcing
# -------------------------------
def wrap_theta(theta): return np.mod(theta, 2.0*np.pi)
def polar_to_cartesian(r, theta): return r*np.cos(theta), r*np.sin(theta)

def annular_forcing_numpy(r, theta):
    ring1 = np.exp(-((r-0.62)/0.13)**2)
    ring2 = np.exp(-((r-0.82)/0.10)**2)
    return ring1*np.sin(3.0*theta+0.25) + 0.35*ring2*np.cos(5.0*theta-0.4)

def annular_initial_numpy(r, theta):
    out = np.zeros_like(r)
    for rc, tc, amp in [(0.55,0.20,2.6), (0.70,1.70,-2.2), (0.58,3.35,2.4), (0.78,5.00,-2.0)]:
        dtheta = np.angle(np.exp(1j*(theta - tc)))
        dist2 = (r-rc)**2 + (rc*dtheta)**2
        out += amp * np.exp(-dist2/(2*0.085**2))
    return out

def annular_forcing_torch(x, y):
    r = torch.sqrt(x.square()+y.square()+1e-12)
    theta = torch.atan2(y, x)
    ring1 = torch.exp(-((r-0.62)/0.13)**2)
    ring2 = torch.exp(-((r-0.82)/0.10)**2)
    return ring1*torch.sin(3.0*theta+0.25) + 0.35*ring2*torch.cos(5.0*theta-0.4)

def annular_initial_torch(x, y):
    r = torch.sqrt(x.square()+y.square()+1e-12)
    theta = torch.atan2(y, x)
    result = torch.zeros_like(r)
    for rc, tc, amp in [(0.55,0.20,2.6), (0.70,1.70,-2.2), (0.58,3.35,2.4), (0.78,5.00,-2.0)]:
        dtheta = torch.atan2(torch.sin(theta-tc), torch.cos(theta-tc))
        dist2 = (r-rc).square() + (rc*dtheta).square()
        result += amp * torch.exp(-dist2/(2*0.085**2))
    return result

def sector_id(theta, n_sectors=4):
    return np.floor(wrap_theta(theta)/(2*np.pi/n_sectors)).astype(int)
def physical_sector_mask(theta): return np.isin(sector_id(theta), [0,2])
def synthetic_sector_mask(theta): return np.isin(sector_id(theta), [1,3])

# -------------------------------
# Finite-difference solver
# -------------------------------
class AnnularVorticitySolver:
    def __init__(self, nr, ntheta, dt, final_time):
        self.nr=nr; self.ntheta=ntheta; self.dt=dt; self.final_time=final_time
        self.r = np.linspace(CFG.r_inner, CFG.r_outer, nr)
        self.theta = np.linspace(0, 2*np.pi, ntheta, endpoint=False)
        self.dr = self.r[1]-self.r[0]; self.dtheta = self.theta[1]-self.theta[0]
        self.R, self.TH = np.meshgrid(self.r, self.theta, indexing="ij")
        self.X, self.Y = polar_to_cartesian(self.R, self.TH)
        self.forcing = annular_forcing_numpy(self.R, self.TH)
        self._build_poisson_solver()
    def _idx(self, ir, it): return (ir-1)*self.ntheta + it
    def _build_poisson_solver(self):
        n_unknowns = (self.nr-2)*self.ntheta
        mat = lil_matrix((n_unknowns, n_unknowns), dtype=float)
        dr2=self.dr**2; dth2=self.dtheta**2
        for ir in range(1, self.nr-1):
            rr = self.r[ir]
            for it in range(self.ntheta):
                row = self._idx(ir, it)
                mat[row,row] = 2.0/dr2 + 2.0/(rr**2*dth2)
                c_r_plus = -(1.0/dr2 + 1.0/(2.0*rr*self.dr))
                c_r_minus = -(1.0/dr2 - 1.0/(2.0*rr*self.dr))
                if ir+1 <= self.nr-2: mat[row, self._idx(ir+1,it)] = c_r_plus
                if ir-1 >= 1: mat[row, self._idx(ir-1,it)] = c_r_minus
                mat[row, self._idx(ir, (it+1)%self.ntheta)] = -1.0/(rr**2*dth2)
                mat[row, self._idx(ir, (it-1)%self.ntheta)] = -1.0/(rr**2*dth2)
        self._poisson = factorized(csc_matrix(mat))
    def solve_streamfunction(self, omega):
        rhs = omega[1:-1,:].reshape(-1)
        psi_inner = self._poisson(rhs)
        psi = np.zeros_like(omega)
        psi[1:-1,:] = psi_inner.reshape(self.nr-2, self.ntheta)
        return psi
    def _derivatives(self, field):
        d_r = np.zeros_like(field)
        d_r[1:-1] = (field[2:]-field[:-2])/(2*self.dr)
        d_r[0] = (field[1]-field[0])/self.dr; d_r[-1] = (field[-1]-field[-2])/self.dr
        d_th = (np.roll(field,-1,axis=1)-np.roll(field,1,axis=1))/(2*self.dtheta)
        d_rr = np.zeros_like(field)
        d_rr[1:-1] = (field[2:]-2*field[1:-1]+field[:-2])/self.dr**2
        d_thth = (np.roll(field,-1,axis=1)-2*field+np.roll(field,1,axis=1))/self.dtheta**2
        lap = d_rr + d_r/self.R + d_thth/self.R**2
        return d_r, d_th, lap
    def rhs(self, omega, nu, rho):
        psi = self.solve_streamfunction(omega)
        psi_r, psi_th, _ = self._derivatives(psi)
        u_r = psi_th/self.R; u_th = -psi_r
        omega_r, omega_th, lap_omega = self._derivatives(omega)
        advection = u_r*omega_r + (u_th/self.R)*omega_th
        result = -advection + nu*lap_omega + rho*self.forcing
        result[0,:] = 0; result[-1,:] = 0
        return result
    def solve(self, nu, rho, store_times):
        store_times = np.asarray(store_times)
        omega = annular_initial_numpy(self.R, self.TH)
        omega[0,:] = 0; omega[-1,:] = 0
        snapshots = np.empty((len(store_times), self.nr, self.ntheta), dtype=np.float32)
        snapshots[0] = omega
        t = 0.0; next_store = 1
        n_steps = int(np.ceil(self.final_time/self.dt))
        for _ in range(n_steps):
            dt = min(self.dt, self.final_time - t)
            if dt <= 0: break
            k1 = self.rhs(omega, nu, rho)
            predictor = omega + dt*k1; predictor[0,:]=0; predictor[-1,:]=0
            k2 = self.rhs(predictor, nu, rho)
            new_omega = omega + 0.5*dt*(k1+k2); new_omega[0,:]=0; new_omega[-1,:]=0
            t_new = t+dt
            while next_store < len(store_times) and store_times[next_store] <= t_new + 1e-12:
                alpha = (store_times[next_store]-t)/max(dt,1e-12)
                snapshots[next_store] = ((1-alpha)*omega + alpha*new_omega).astype(np.float32)
                next_store += 1
            omega = new_omega; t = t_new
        while next_store < len(store_times):
            snapshots[next_store] = omega.astype(np.float32); next_store += 1
        return snapshots

# -------------------------------
# Sampling and interpolation
# -------------------------------
def sample_annulus_points(n, store_times, mask_fn=None, include_initial=False):
    points = []
    candidate_times = store_times if include_initial else store_times[1:]
    while len(points) < n:
        m = max(256, 2*(n-len(points)))
        u = np.random.rand(m)
        r = np.sqrt(CFG.r_inner**2 + u*(CFG.r_outer**2 - CFG.r_inner**2))
        theta = 2*np.pi*np.random.rand(m)
        if mask_fn is not None:
            keep = mask_fn(theta); r, theta = r[keep], theta[keep]
        t = np.random.choice(candidate_times, size=len(r), replace=True)
        points.extend(zip(r.tolist(), theta.tolist(), t.tolist()))
    return np.asarray(points[:n], dtype=np.float64)

def sample_cube_polar(cube, r_grid, theta_grid, times, points):
    r = np.clip(points[:,0], r_grid[0], r_grid[-1])
    theta = wrap_theta(points[:,1])
    t = np.clip(points[:,2], times[0], times[-1])
    ir1 = np.searchsorted(r_grid, r, side="right")
    ir1 = np.clip(ir1, 1, len(r_grid)-1); ir0 = ir1-1
    ar = (r-r_grid[ir0])/(r_grid[ir1]-r_grid[ir0]+1e-12)
    dtheta = theta_grid[1]-theta_grid[0]
    pos = theta/dtheta
    it0 = np.floor(pos).astype(int) % len(theta_grid); it1 = (it0+1)%len(theta_grid)
    ath = pos - np.floor(pos)
    kt1 = np.searchsorted(times, t, side="right")
    kt1 = np.clip(kt1, 1, len(times)-1); kt0 = kt1-1
    at = (t-times[kt0])/(times[kt1]-times[kt0]+1e-12)
    v000 = cube[kt0, ir0, it0]; v001 = cube[kt0, ir0, it1]
    v010 = cube[kt0, ir1, it0]; v011 = cube[kt0, ir1, it1]
    v100 = cube[kt1, ir0, it0]; v101 = cube[kt1, ir0, it1]
    v110 = cube[kt1, ir1, it0]; v111 = cube[kt1, ir1, it1]
    v0 = (1-ar)*((1-ath)*v000 + ath*v001) + ar*((1-ath)*v010 + ath*v011)
    v1 = (1-ar)*((1-ath)*v100 + ath*v101) + ar*((1-ath)*v110 + ath*v111)
    return (1-at)*v0 + at*v1

def points_polar_to_network(points):
    x,y = polar_to_cartesian(points[:,0], points[:,1])
    t = points[:,2]/CFG.final_time
    return np.column_stack([x,y,t]).astype(np.float32)

def add_relative_noise(values, level):
    if level <= 0: return values.copy()
    sigma = level * np.std(values)
    return values + sigma*np.random.randn(*values.shape)

# Generate reference solution
store_times = np.linspace(0.0, CFG.final_time, CFG.n_store)
true_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true, CFG.dt_true, CFG.final_time)
physical_solver = AnnularVorticitySolver(CFG.nr_phy, CFG.ntheta_phy, CFG.dt_phy, CFG.final_time)

truth_path = ROOT / "data" / "reference_solution.npz"
if truth_path.exists():
    payload = np.load(truth_path); truth_cube = payload["omega"]
else:
    print("Generating reference solution...")
    truth_cube = true_solver.solve(CFG.nu_true, CFG.rho_true, store_times)
    np.savez_compressed(truth_path, omega=truth_cube, r=true_solver.r, theta=true_solver.theta, times=store_times)

# Generate fixed ghost points (for all subsequent experiments)
obs_phy_pts = sample_annulus_points(CFG.n_obs_per_agent, store_times, physical_sector_mask)
obs_syn_pts = sample_annulus_points(CFG.n_obs_per_agent, store_times, synthetic_sector_mask)
validation_pts = sample_annulus_points(CFG.n_validation, store_times)
ghost_pts = sample_annulus_points(CFG.n_ghost, store_times)
ghost_test_pts = sample_annulus_points(max(800, CFG.n_ghost), store_times)

# Save these fixed points (for later use)
np.savez_compressed(ROOT/"data"/"fixed_points.npz",
    obs_phy_pts=obs_phy_pts, obs_syn_pts=obs_syn_pts,
    validation_pts=validation_pts, ghost_pts=ghost_pts, ghost_test_pts=ghost_test_pts)

# Define neural networks
class FourierFeatures(nn.Module):
    def __init__(self, spatial_freq=(1,2,3,5), temporal_freq=(1,2,4)):
        super().__init__()
        self.spatial = spatial_freq
        self.temporal = temporal_freq

    @property
    def out_dim(self):
        """Total number of features: x,y,t + sin/cos for each spatial freq + sin/cos for each temporal freq."""
        return 3 + 4 * len(self.spatial) + 2 * len(self.temporal)

    def forward(self, z):
        x, y, t = z[:, 0:1], z[:, 1:2], z[:, 2:3]
        features = [x, y, t]
        for k in self.spatial:
            features.extend([
                torch.sin(math.pi * k * x),
                torch.cos(math.pi * k * x),
                torch.sin(math.pi * k * y),
                torch.cos(math.pi * k * y),
            ])
        for k in self.temporal:
            features.extend([
                torch.sin(2.0 * math.pi * k * t),
                torch.cos(2.0 * math.pi * k * t),
            ])
        return torch.cat(features, dim=1)

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width, depth, activation=nn.Tanh):
        super().__init__()
        layers = [nn.Linear(in_dim,width), activation()]
        for _ in range(depth-1): layers += [nn.Linear(width,width), activation()]
        layers.append(nn.Linear(width,out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()
    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

class SyntheticVorticityNet(nn.Module):
    def __init__(self, width, depth):
        super().__init__()
        self.features = FourierFeatures()
        self.mlp = MLP(self.features.out_dim, 1, width, depth)
    def forward(self, z): return self.mlp(self.features(z))

class MixedPINN(nn.Module):
    def __init__(self, width, depth):
        super().__init__()
        self.features = FourierFeatures()
        self.mlp = MLP(self.features.out_dim, 2, width, depth)
    def forward(self, z): return self.mlp(self.features(z))

class SectorXPINN(nn.Module):
    def __init__(self, width, depth, n_sectors=4):
        super().__init__()
        self.n_sectors = n_sectors
        self.models = nn.ModuleList([MixedPINN(width, depth) for _ in range(n_sectors)])
    def sector_tensor(self, z):
        theta = torch.atan2(z[:,1], z[:,0])
        theta = torch.remainder(theta, 2*math.pi)
        return torch.floor(theta/(2*math.pi/self.n_sectors)).long().clamp(0, self.n_sectors-1)
    def forward(self, z):
        sectors = self.sector_tensor(z)
        output = torch.empty((len(z),2), dtype=z.dtype, device=z.device)
        for k, model in enumerate(self.models):
            mask = sectors==k
            if torch.any(mask): output[mask] = model(z[mask])
        return output

def to_tensor(array, requires_grad=False):
    return torch.tensor(array, dtype=DTYPE, device=DEVICE, requires_grad=requires_grad)

def grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs),
                               create_graph=True, retain_graph=True)[0]

def mixed_pinn_residual(model, z, nu, rho):
    z = z.requires_grad_(True)
    out = model(z); psi = out[:,0:1]; omega = out[:,1:2]
    gpsi = grad(psi,z); psi_x=gpsi[:,0:1]; psi_y=gpsi[:,1:2]
    psi_xx = grad(psi_x,z)[:,0:1]; psi_yy = grad(psi_y,z)[:,1:2]
    gomega = grad(omega,z); omega_x=gomega[:,0:1]; omega_y=gomega[:,1:2]
    omega_t = gomega[:,2:3]/CFG.final_time
    omega_xx = grad(omega_x,z)[:,0:1]; omega_yy = grad(omega_y,z)[:,1:2]
    x,y = z[:,0:1], z[:,1:2]
    forcing = annular_forcing_torch(x,y)
    u = psi_y; v = -psi_x
    r_transport = omega_t + u*omega_x + v*omega_y - nu*(omega_xx+omega_yy) - rho*forcing
    r_poisson = omega + psi_xx + psi_yy
    return r_transport, r_poisson, psi, omega

def sample_collocation_annulus(n):
    u = np.random.rand(n)
    r = np.sqrt(CFG.r_inner**2 + u*(CFG.r_outer**2 - CFG.r_inner**2))
    theta = 2*np.pi*np.random.rand(n)
    x,y = polar_to_cartesian(r, theta)
    t = np.random.rand(n)
    return np.column_stack([x,y,t]).astype(np.float32)

def sample_ic_annulus(n):
    z = sample_collocation_annulus(n); z[:,2]=0.0; return z

def sample_bc_annulus(n):
    n_half = n//2
    theta = 2*np.pi*np.random.rand(n)
    r = np.concatenate([np.full(n_half, CFG.r_inner), np.full(n-n_half, CFG.r_outer)])
    x,y = polar_to_cartesian(r, theta)
    t = np.random.rand(n)
    return np.column_stack([x,y,t]).astype(np.float32)

def evaluate_network_numpy(model, points_network, output_index=0):
    model.eval()
    output = []
    with torch.no_grad():
        for start in range(0, len(points_network), CFG.eval_batch):
            batch = to_tensor(points_network[start:start+CFG.eval_batch])
            pred = model(batch)
            if pred.ndim==2 and pred.shape[1]>1:
                pred = pred[:, output_index:output_index+1]
            output.append(pred.detach().cpu().numpy().reshape(-1))
    return np.concatenate(output)

print("Block 1 executed successfully.")

2. Main Experiment

In [ ]:
# ================================
# Block 2 (final alignment): HYCO training with interaction loss, no PINN residual,
# independent local updates, constrained parameter averaging, and return last-round results
# ================================
import time, math
from scipy.optimize import minimize
from pathlib import Path
import pandas as pd
import torch
import numpy as np

def train_hyco(obs_phy_pts, obs_phy_y, obs_syn_pts, obs_syn_y,
               validation_pts, validation_y, noise_level, output_root):
    """
    HYCO training:
    - Both agents start from the same global state (model + nu, rho).
    - Physical agent: L-BFGS for nu, rho (no cache); network trained with data + interaction (no IC/BC/PDE).
    - Synthetic agent: network trained with data + IC + BC + interaction (no PDE).
    - Ghost points resampled every round.
    - Aggregation: FedAvg on network weights and nu/rho.
    - Global nu/rho are clipped to the admissible physical bounds after averaging.
    - Returns the final-round results (not best validation), aligning with convergence plots.
    """
    root = Path(output_root)
    for sub in ["data", "logs", "checkpoints"]:
        (root / sub).mkdir(parents=True, exist_ok=True)

    # ---------- Hyperparameters (Appendix C) ----------
    SYN_STEPS = 100
    PHYS_NET_STEPS = 100
    PHYS_LBFGS_ITER = 20
    IC_NUM = 400
    BC_NUM = 400
    GHOST_NUM = CFG.n_ghost        # 3000
    LR_NET = 1e-3

    # Loss weights
    W_DATA_PHY = 1.0
    W_DATA_SYN = 4.0
    W_IC = 2.0
    W_BC = 0.5
    W_INTERACTION = 2.0

    # ---------- Agent-specific data (fixed) ----------
    phys_z = to_tensor(points_polar_to_network(obs_phy_pts))
    phys_y = to_tensor(obs_phy_y[:, None].astype(np.float32))
    syn_z = to_tensor(points_polar_to_network(obs_syn_pts))
    syn_y = to_tensor(obs_syn_y[:, None].astype(np.float32))

    # ---------- Global state ----------
    global_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
    global_nu = torch.tensor(0.5, dtype=DTYPE, device=DEVICE, requires_grad=False)
    global_rho = torch.tensor(0.5, dtype=DTYPE, device=DEVICE, requires_grad=False)

    history = []
    start_time = time.perf_counter()

    for round_idx in range(CFG.hyco_rounds):
        # --- 0. Sample IC, BC and ghost points ---
        ic_pts_polar = sample_annulus_points(IC_NUM, np.asarray([0.0, 1.0]), include_initial=True)
        ic_pts_polar[:, 2] = 0.0
        ic_z = to_tensor(points_polar_to_network(ic_pts_polar))
        ic_target = annular_initial_torch(ic_z[:, 0:1], ic_z[:, 1:2]).detach()

        bc_np = sample_bc_annulus(BC_NUM)
        bc_z = to_tensor(bc_np)
        bc_target = torch.zeros((len(bc_np), 1), dtype=DTYPE, device=DEVICE)

        ghost_pts = sample_annulus_points(GHOST_NUM, store_times)
        ghost_z = to_tensor(points_polar_to_network(ghost_pts))

        # --- 1. Broadcast global state to agents ---
        model_phys = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
        model_phys.load_state_dict(global_model.state_dict())
        model_syn = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
        model_syn.load_state_dict(global_model.state_dict())

        nu_phys_param = torch.nn.Parameter(global_nu.clone().detach())
        rho_phys_param = torch.nn.Parameter(global_rho.clone().detach())
        # For synthetic we also keep trainable parameters (though not used in loss, we keep for symmetry)
        nu_syn_param = torch.nn.Parameter(global_nu.clone().detach())
        rho_syn_param = torch.nn.Parameter(global_rho.clone().detach())

        # --- 2. Physical agent: L-BFGS for nu, rho (no cache) ---
        def objective_phys(log_params):
            nu = float(np.exp(log_params[0]))
            rho = float(np.exp(log_params[1]))
            cube = physical_solver.solve(nu, rho, store_times)
            pred = sample_cube_polar(cube, physical_solver.r, physical_solver.theta,
                                     store_times, obs_phy_pts)
            data_loss = np.mean((pred - obs_phy_y) ** 2)
            reg = 1e-5 * ((nu / CFG.nu_true) ** 2 + (rho / CFG.rho_true) ** 2)
            return data_loss + reg

        bounds = [(math.log(CFG.nu_min), math.log(CFG.nu_max)),
                  (math.log(CFG.rho_min), math.log(CFG.rho_max))]
        res = minimize(objective_phys,
                       x0=np.log([nu_phys_param.item(), rho_phys_param.item()]),
                       method="L-BFGS-B", bounds=bounds,
                       options={"maxiter": PHYS_LBFGS_ITER, "ftol": 1e-10})
        nu_phys_opt = float(np.exp(res.x[0]))
        rho_phys_opt = float(np.exp(res.x[1]))

        # --- 3. Physical network training (data + interaction) ---
        # Use synthetic ghost targets from model_syn (which is still the broadcast copy)
        with torch.no_grad():
            ghost_syn_target = model_syn(ghost_z)[:, 1:2].detach()

        optimizer_phys_net = torch.optim.Adam(model_phys.parameters(), lr=LR_NET)
        model_phys.train()
        for _ in range(PHYS_NET_STEPS):
            optimizer_phys_net.zero_grad()
            pred_data = model_phys(phys_z)[:, 1:2]
            loss_data = torch.mean((pred_data - phys_y) ** 2)
            pred_ghost = model_phys(ghost_z)[:, 1:2]
            loss_inter = torch.mean((pred_ghost - ghost_syn_target) ** 2)
            total_loss = W_DATA_PHY * loss_data + W_INTERACTION * loss_inter
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model_phys.parameters(), 5.0)
            optimizer_phys_net.step()

        # --- 4. Synthetic network training (data + IC + BC + interaction) ---
        # Ghost target: physical solver prediction (not the physical network)
        cube_phys = physical_solver.solve(nu_phys_opt, rho_phys_opt, store_times)
        ghost_phy_target_np = sample_cube_polar(cube_phys, physical_solver.r, physical_solver.theta,
                                                store_times, ghost_pts).astype(np.float32)
        ghost_phy_target = to_tensor(ghost_phy_target_np[:, None])

        optimizer_syn_net = torch.optim.Adam(model_syn.parameters(), lr=LR_NET)
        model_syn.train()
        for _ in range(SYN_STEPS):
            optimizer_syn_net.zero_grad()
            pred_data = model_syn(syn_z)[:, 1:2]
            loss_data = torch.mean((pred_data - syn_y) ** 2)
            pred_ic = model_syn(ic_z)[:, 1:2]
            loss_ic = torch.mean((pred_ic - ic_target) ** 2)
            pred_bc = model_syn(bc_z)[:, 1:2]
            loss_bc = torch.mean((pred_bc - bc_target) ** 2)
            pred_ghost = model_syn(ghost_z)[:, 1:2]
            loss_inter = torch.mean((pred_ghost - ghost_phy_target) ** 2)
            total_loss = (W_DATA_SYN * loss_data +
                          W_IC * loss_ic +
                          W_BC * loss_bc +
                          W_INTERACTION * loss_inter)
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model_syn.parameters(), 5.0)
            optimizer_syn_net.step()

        # --- 5. Aggregation (FedAvg) with clipping to admissible bounds ---
        avg_state = {}
        for key in global_model.state_dict().keys():
            avg_state[key] = (model_phys.state_dict()[key] + model_syn.state_dict()[key]) / 2.0
        global_model.load_state_dict(avg_state)

        avg_nu = (nu_phys_opt + nu_syn_param.item()) / 2.0
        avg_rho = (rho_phys_opt + rho_syn_param.item()) / 2.0
        # Clip to physical bounds
        avg_nu = np.clip(avg_nu, CFG.nu_min, CFG.nu_max)
        avg_rho = np.clip(avg_rho, CFG.rho_min, CFG.rho_max)
        global_nu.data.fill_(avg_nu)
        global_rho.data.fill_(avg_rho)

        # --- 6. Validation (for logging only) ---
        global_model.eval()
        val_pred = evaluate_network_numpy(global_model,
                                          points_polar_to_network(validation_pts),
                                          output_index=1)
        score = float(np.mean((val_pred - validation_y) ** 2))
        history.append({
            "round": round_idx,
            "nu": avg_nu,
            "rho": avg_rho,
            "validation_score": score
        })

        if (round_idx + 1) % 10 == 0 or round_idx == CFG.hyco_rounds - 1:
            print(f"Round {round_idx+1:3d}/{CFG.hyco_rounds:3d} | "
                  f"nu={avg_nu:.8e} rho={avg_rho:.8f} | val={score:.3e}")

    runtime = time.perf_counter() - start_time

    # Return the final-round results (not best validation)
    final_model = global_model   # global_model is the aggregated state from last round
    final_nu = float(global_nu.cpu().numpy())
    final_rho = float(global_rho.cpu().numpy())

    torch.save(final_model.state_dict(), root / "checkpoints" / "hyco_synthetic.pt")
    if history:
        history[-1]["runtime"] = runtime
    pd.DataFrame(history).to_csv(root / "logs" / "hyco_history.csv", index=False)
    np.savez(root / "data" / "hyco_results.npz", nu=final_nu, rho=final_rho, runtime=runtime)

    return {"nu": final_nu, "rho": final_rho, "runtime": runtime, "history": history}
        

# ------------------------------- Main loop -------------------------------
if CFG.run_hyco:
    noise_levels = [0.00, 0.05, 0.10, 0.15, 0.20] # [0.00, 0.05, 0.10, 0.15, 0.20]
    fixed = np.load(ROOT/"data"/"fixed_points.npz")
    obs_phy_pts_base = fixed["obs_phy_pts"]
    obs_syn_pts_base = fixed["obs_syn_pts"]
    validation_pts = fixed["validation_pts"]
    
    truth = np.load(ROOT/"data"/"reference_solution.npz")
    truth_cube = truth["omega"]
    r_grid_true = truth["r"]
    theta_grid_true = truth["theta"]
    store_times = truth["times"]
    
    for nl in noise_levels:
        print(f"\n=== HYCO noise level {nl:.2f} ===")
        obs_phy_pts = obs_phy_pts_base
        obs_syn_pts = obs_syn_pts_base
        obs_phy_y_clean = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, obs_phy_pts)
        obs_syn_y_clean = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, obs_syn_pts)
        validation_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, validation_pts)
        obs_phy_y = add_relative_noise(obs_phy_y_clean, nl)
        obs_syn_y = add_relative_noise(obs_syn_y_clean, nl)
        
        output_root = ROOT / "hyco" / f"noise_{int(nl*100):02d}"
        result = train_hyco(obs_phy_pts, obs_phy_y, obs_syn_pts, obs_syn_y,
                            validation_pts, validation_y, nl, output_root)
        print(f"Finished noise {nl:.2f}: nu={result['nu']:.4e}, rho={result['rho']:.5f}, runtime={result['runtime']:.1f}s")

3. PINN

In [ ]:
# ================================
# Block 3: Standard PINN (noise=0.00)  with hard bounds & init (0.5, 0.5)
# ================================
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path

def train_pinn():
    pinn_root = ROOT / "pinn"
    for sub in ["logs", "checkpoints", "data"]:
        (pinn_root/sub).mkdir(parents=True, exist_ok=True)
    
    # Load noise-free observation data
    fixed = np.load(ROOT/"data"/"fixed_points.npz")
    obs_phy_pts = fixed["obs_phy_pts"]
    obs_syn_pts = fixed["obs_syn_pts"]
    validation_pts = fixed["validation_pts"]
    truth = np.load(ROOT/"data"/"reference_solution.npz")
    truth_cube = truth["omega"]
    r_grid_true = truth["r"]; theta_grid_true = truth["theta"]; store_times = truth["times"]
    obs_phy_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, obs_phy_pts)
    obs_syn_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, obs_syn_pts)
    validation_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, validation_pts)
    data_pts = np.vstack([obs_phy_pts, obs_syn_pts])
    data_y = np.concatenate([obs_phy_y, obs_syn_y]).astype(np.float32)
    data_z = points_polar_to_network(data_pts)
    
    model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
    
    # Parameters initialized to 0.5 (will be projected to feasible range)
    nu_param = nn.Parameter(torch.tensor(0.5, dtype=DTYPE, device=DEVICE))
    rho_param = nn.Parameter(torch.tensor(0.5, dtype=DTYPE, device=DEVICE))
    
    # Hard bounds
    nu_min = CFG.nu_min
    nu_max = CFG.nu_max
    rho_min = CFG.rho_min
    rho_max = CFG.rho_max
    
    def project_params():
        """Project parameters to the admissible set."""
        with torch.no_grad():
            nu_param.clamp_(nu_min, nu_max)
            rho_param.clamp_(rho_min, rho_max)
    
    optimizer = torch.optim.Adam(list(model.parameters()) + [nu_param, rho_param], lr=8e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(250, CFG.pinn_epochs//4), gamma=0.75)
    
    best_score = np.inf
    best_state = None
    history = []
    start_time = time.perf_counter()
    
    for epoch in range(CFG.pinn_epochs):
        optimizer.zero_grad(set_to_none=True)
        
        # Get current feasible parameters (they are already inside bounds after projection)
        nu = nu_param
        rho = rho_param
        
        # Data loss
        idx = np.random.choice(len(data_z), size=min(CFG.batch_data, len(data_z)), replace=False)
        z_data = to_tensor(data_z[idx])
        y_data = to_tensor(data_y[idx, None])
        pred_data = model(z_data)[:,1:2]
        loss_data = torch.mean((pred_data - y_data)**2)
        
        # Collocation points for PDE
        z_col = to_tensor(sample_collocation_annulus(CFG.n_collocation), requires_grad=True)
        r_pde, r_poisson, _, _ = mixed_pinn_residual(model, z_col, nu, rho)
        loss_pde = torch.mean(r_pde**2)
        loss_poisson = torch.mean(r_poisson**2)
        
        # Initial condition
        z_ic = to_tensor(sample_ic_annulus(CFG.n_ic))
        omega_ic = model(z_ic)[:,1:2]
        target_ic = annular_initial_torch(z_ic[:,0:1], z_ic[:,1:2]).detach()
        loss_ic = torch.mean((omega_ic - target_ic)**2)
        
        # Boundary condition (ψ=0 on both walls, but our model outputs (ψ,ω); we impose ω=0? Actually BC is ψ=0, but we force both outputs? 
        # Original code used model(z_bc) squared, which imposes both outputs zero. We keep as is.)
        z_bc = to_tensor(sample_bc_annulus(CFG.n_bc))
        loss_bc = torch.mean(model(z_bc)**2)
        
        loss = (CFG.w_pinn_data*loss_data + CFG.w_pinn_pde*loss_pde +
                CFG.w_pinn_poisson*loss_poisson + CFG.w_pinn_ic*loss_ic + CFG.w_pinn_bc*loss_bc)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + [nu_param, rho_param], 10.0)
        optimizer.step()
        
        # Apply hard bounds
        project_params()
        
        scheduler.step()
        
        # Logging and best model saving
        if epoch % 50 == 0 or epoch == CFG.pinn_epochs-1:
            with torch.no_grad():
                nu_val = nu_param.item()
                rho_val = rho_param.item()
            val_pred = evaluate_network_numpy(model, points_polar_to_network(validation_pts), output_index=1)
            score = float(np.mean((val_pred - validation_y)**2))
            history.append({"epoch": epoch, "nu": nu_val, "rho": rho_val, "validation_mse": score})
            if score < best_score:
                best_score = score
                best_state = {
                    "network": {k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                    "nu_param": nu_param.detach().cpu().clone(),
                    "rho_param": rho_param.detach().cpu().clone()
                }
            if epoch % 500 == 0:
                print(f"PINN epoch {epoch}: nu={nu_val:.8e}, rho={rho_val:.8f}, val={score:.3e}")
    
    runtime = time.perf_counter() - start_time
    assert best_state is not None
    model.load_state_dict(best_state["network"])
    final_nu = float(best_state["nu_param"].item())
    final_rho = float(best_state["rho_param"].item())
    torch.save({"model": model.state_dict(), "nu_param": best_state["nu_param"], "rho_param": best_state["rho_param"]},
               pinn_root/"checkpoints"/"pinn.pt")
    if history:
        history[-1]["runtime"] = runtime
    pd.DataFrame(history).to_csv(pinn_root/"logs"/"pinn_history.csv", index=False)
    np.savez(pinn_root/"data"/"pinn_results.npz", nu=final_nu, rho=final_rho, runtime=runtime)
    return {"nu": final_nu, "rho": final_rho, "runtime": runtime, "history": history}

if CFG.run_pinn:
    pinn_res = train_pinn()
    print(f"PINN finished: nu={pinn_res['nu']:.4e}, rho={pinn_res['rho']:.5f}, runtime={pinn_res['runtime']:.1f}s")
else:
    print("PINN skipped (set run_pinn=True to run).")

4. XPINN

In [ ]:
# ================================
# Block 4: XPINN Training (noise=0.00) with hard bounds & init (0.5, 0.5)
# ================================
import time, math
import torch
import numpy as np
import pandas as pd
from pathlib import Path

def sample_interface_annulus(n_per_interface):
    pairs = []
    eps = 2e-4
    for k in range(4):
        theta0 = k * 0.5 * np.pi
        r = np.sqrt(CFG.r_inner**2 + np.random.rand(n_per_interface)*(CFG.r_outer**2 - CFG.r_inner**2))
        t = np.random.rand(n_per_interface)
        left_theta = np.full(n_per_interface, theta0 - eps)
        right_theta = np.full(n_per_interface, theta0 + eps)
        xl, yl = polar_to_cartesian(r, left_theta)
        xr, yr = polar_to_cartesian(r, right_theta)
        zl = np.column_stack([xl, yl, t]).astype(np.float32)
        zr = np.column_stack([xr, yr, t]).astype(np.float32)
        left_id = (k-1) % 4
        right_id = k % 4
        pairs.append((zl, zr, left_id, right_id))
    return pairs

def train_xpinn():
    xpinn_root = ROOT / "xpinn"
    for sub in ["logs", "checkpoints", "data"]:
        (xpinn_root/sub).mkdir(parents=True, exist_ok=True)
    
    # Data (same as PINN)
    fixed = np.load(ROOT/"data"/"fixed_points.npz")
    obs_phy_pts = fixed["obs_phy_pts"]; obs_syn_pts = fixed["obs_syn_pts"]
    validation_pts = fixed["validation_pts"]
    truth = np.load(ROOT/"data"/"reference_solution.npz")
    truth_cube = truth["omega"]; r_grid_true = truth["r"]; theta_grid_true = truth["theta"]; store_times = truth["times"]
    obs_phy_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, obs_phy_pts)
    obs_syn_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, obs_syn_pts)
    validation_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, validation_pts)
    data_pts = np.vstack([obs_phy_pts, obs_syn_pts])
    data_y = np.concatenate([obs_phy_y, obs_syn_y]).astype(np.float32)
    data_z = points_polar_to_network(data_pts)
    
    model = SectorXPINN(max(56, CFG.nn_width//1), CFG.nn_depth, 4).to(DEVICE)
    
    # Parameters initialized to 0.5
    nu_param = nn.Parameter(torch.tensor(0.5, dtype=DTYPE, device=DEVICE))
    rho_param = nn.Parameter(torch.tensor(0.5, dtype=DTYPE, device=DEVICE))
    
    # Hard bounds
    nu_min = CFG.nu_min
    nu_max = CFG.nu_max
    rho_min = CFG.rho_min
    rho_max = CFG.rho_max
    
    def project_params():
        with torch.no_grad():
            nu_param.clamp_(nu_min, nu_max)
            rho_param.clamp_(rho_min, rho_max)
    
    optimizer = torch.optim.Adam(list(model.parameters()) + [nu_param, rho_param], lr=8e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(300, CFG.xpinn_epochs//4), gamma=0.75)
    
    best_score = np.inf
    best_state = None
    history = []
    start_time = time.perf_counter()
    
    for epoch in range(CFG.xpinn_epochs):
        optimizer.zero_grad(set_to_none=True)
        
        nu = nu_param
        rho = rho_param
        
        # Data loss
        idx = np.random.choice(len(data_z), size=min(CFG.batch_data, len(data_z)), replace=False)
        z_data = to_tensor(data_z[idx]); y_data = to_tensor(data_y[idx, None])
        pred_data = model(z_data)[:,1:2]
        loss_data = torch.mean((pred_data - y_data)**2)
        
        # PDE loss per sector
        z_col_np = sample_collocation_annulus(CFG.n_collocation)
        z_col = to_tensor(z_col_np)
        sector_labels = model.sector_tensor(z_col)
        pde_terms = []
        poisson_terms = []
        for k, local_model in enumerate(model.models):
            mask = sector_labels == k
            if torch.any(mask):
                z_local = z_col[mask].detach().clone().requires_grad_(True)
                r_pde, r_poisson, _, _ = mixed_pinn_residual(local_model, z_local, nu, rho)
                pde_terms.append(torch.mean(r_pde**2))
                poisson_terms.append(torch.mean(r_poisson**2))
        loss_pde = torch.stack(pde_terms).mean() if pde_terms else torch.tensor(0.0, device=DEVICE)
        loss_poisson = torch.stack(poisson_terms).mean() if poisson_terms else torch.tensor(0.0, device=DEVICE)
        
        # IC and BC
        z_ic = to_tensor(sample_ic_annulus(CFG.n_ic))
        omega_ic = model(z_ic)[:,1:2]
        target_ic = annular_initial_torch(z_ic[:,0:1], z_ic[:,1:2]).detach()
        loss_ic = torch.mean((omega_ic - target_ic)**2)
        z_bc = to_tensor(sample_bc_annulus(CFG.n_bc))
        loss_bc = torch.mean(model(z_bc)**2)
        
        # Interface loss
        interface_losses = []
        for zl_np, zr_np, left_id, right_id in sample_interface_annulus(max(40, CFG.n_interface//4)):
            zl = to_tensor(zl_np); zr = to_tensor(zr_np)
            out_l = model.models[left_id](zl)
            out_r = model.models[right_id](zr)
            interface_losses.append(torch.mean((out_l - out_r)**2))
        loss_interface = torch.stack(interface_losses).mean() if interface_losses else torch.tensor(0.0, device=DEVICE)
        
        loss = (CFG.w_pinn_data*loss_data + CFG.w_pinn_pde*loss_pde +
                CFG.w_pinn_poisson*loss_poisson + CFG.w_pinn_ic*loss_ic +
                CFG.w_pinn_bc*loss_bc + CFG.w_xpinn_interface*loss_interface)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + [nu_param, rho_param], 10.0)
        optimizer.step()
        
        # Project parameters
        project_params()
        
        scheduler.step()
        
        if epoch % 100 == 0 or epoch == CFG.xpinn_epochs-1:
            with torch.no_grad():
                nu_val = nu_param.item()
                rho_val = rho_param.item()
            val_pred = evaluate_network_numpy(model, points_polar_to_network(validation_pts), output_index=1)
            score = float(np.mean((val_pred - validation_y)**2))
            history.append({"epoch": epoch, "nu": nu_val, "rho": rho_val, "validation_mse": score,
                            "interface_loss": float(loss_interface.detach().cpu())})
            if score < best_score:
                best_score = score
                best_state = {
                    "network": {k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                    "nu_param": nu_param.detach().cpu().clone(),
                    "rho_param": rho_param.detach().cpu().clone()
                }
            if epoch % 500 == 0:
                print(f"XPINN epoch {epoch}: nu={nu_val:.8e}, rho={rho_val:.8f}, val={score:.3e}")
    
    runtime = time.perf_counter() - start_time
    assert best_state is not None
    model.load_state_dict(best_state["network"])
    final_nu = float(best_state["nu_param"].item())
    final_rho = float(best_state["rho_param"].item())
    torch.save({"model": model.state_dict(), "nu_param": best_state["nu_param"], "rho_param": best_state["rho_param"]},
               xpinn_root/"checkpoints"/"xpinn.pt")
    if history:
        history[-1]["runtime"] = runtime
    pd.DataFrame(history).to_csv(xpinn_root/"logs"/"xpinn_history.csv", index=False)
    np.savez(xpinn_root/"data"/"xpinn_results.npz", nu=final_nu, rho=final_rho, runtime=runtime)
    return {"nu": final_nu, "rho": final_rho, "runtime": runtime, "history": history}

if CFG.run_xpinn:
    xpinn_res = train_xpinn()
    print(f"XPINN finished: nu={xpinn_res['nu']:.4e}, rho={xpinn_res['rho']:.5f}, runtime={xpinn_res['runtime']:.1f}s")
else:
    print("XPINN skipped (set run_xpinn=True to run).")

5. Plot and Tables

In [ ]:
# ================================
# Block 5 (final revision): HYCO Multi-Noise Plotting and Table
# - Correct relative L2 errors for PHY and SYN
# - Dynamic zoom subplot tick labels
# - Rename interaction errors to e_int_eval
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator
from matplotlib.patches import ConnectionPatch
import torch
from pathlib import Path

# ================== 1. Load HYCO results for all noise levels ==================
noise_levels = [0.00, 0.05, 0.10, 0.15, 0.20]
all_results = []
for nl in noise_levels:
    subdir = ROOT / "hyco" / f"noise_{int(nl*100):02d}"
    if not (subdir / "logs" / "hyco_history.csv").exists():
        print(f"Warning: {subdir} not complete, skip.")
        continue
    hist = pd.read_csv(subdir / "logs" / "hyco_history.csv")
    res = np.load(subdir / "data" / "hyco_results.npz")
    nu_est = res["nu"].item()
    rho_est = res["rho"].item()
    runtime = res["runtime"].item()
    all_results.append({
        "noise": nl,
        "history": hist,
        "nu": nu_est,
        "rho": rho_est,
        "runtime": runtime,
        "subdir": subdir
    })

if len(all_results) == 0:
    raise RuntimeError("No HYCO results found.")

# ================== 2. Compute relative L2 errors for each noise level ==================
# Load ground truth and validation points
fixed = np.load(ROOT / "data" / "fixed_points.npz")
validation_pts = fixed["validation_pts"]
truth = np.load(ROOT / "data" / "reference_solution.npz")
truth_cube = truth["omega"]
r_grid_true = truth["r"]
theta_grid_true = truth["theta"]
store_times = truth["times"]
validation_y = sample_cube_polar(truth_cube, r_grid_true, theta_grid_true, store_times, validation_pts)

def rel_l2_error(pred, true):
    return np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-12)

for res in all_results:
    nl = res["noise"]
    nu_est = res["nu"]
    rho_est = res["rho"]
    subdir = res["subdir"]

    # Load synthetic network
    model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
    model.load_state_dict(torch.load(subdir / "checkpoints" / "hyco_synthetic.pt", map_location=DEVICE))
    model.eval()

    # HYCO-PHY (physical solver)
    cube_phys = physical_solver.solve(nu_est, rho_est, store_times)
    phys_vals = sample_cube_polar(cube_phys, physical_solver.r, physical_solver.theta,
                                  store_times, validation_pts)
    # HYCO-SYN (neural network)
    z_val = points_polar_to_network(validation_pts)
    syn_vals = evaluate_network_numpy(model, z_val, output_index=1)

    res["rel_l2_phy"] = rel_l2_error(phys_vals, validation_y)
    res["rel_l2_syn"] = rel_l2_error(syn_vals, validation_y)

# ================== 3. Plot convergence curves (with dynamic zoom labels) ==================
nu_seqs = []
rho_seqs = []
rounds_list = []

for res in all_results:
    hist = res["history"]
    rounds_orig = hist["round"].values
    nu_orig = hist["nu"].values
    rho_orig = hist["rho"].values
    # Insert initial values at round 0 (nu=0.5, rho=0.5)
    rounds_new = np.insert(rounds_orig, 0, 0)
    nu_new = np.insert(nu_orig, 0, 0.5)
    rho_new = np.insert(rho_orig, 0, 0.5)
    nu_seqs.append(nu_new)
    rho_seqs.append(rho_new)
    rounds_list.append(rounds_new)

# Truncate to shortest length
min_len = min(len(s) for s in nu_seqs)
nu_seqs = [seq[:min_len] for seq in nu_seqs]
rho_seqs = [seq[:min_len] for seq in rho_seqs]
common_rounds = rounds_list[0][:min_len]   # includes 0

nu_arr = np.array(nu_seqs)
rho_arr = np.array(rho_seqs)
nu_mean = np.mean(nu_arr, axis=0)
nu_std = np.std(nu_arr, axis=0, ddof=1) if len(nu_arr) >= 2 else np.zeros_like(nu_mean)
rho_mean = np.mean(rho_arr, axis=0)
rho_std = np.std(rho_arr, axis=0, ddof=1) if len(rho_arr) >= 2 else np.zeros_like(rho_mean)

# Zoom region: last N_ZOOM points
N_ZOOM = 5
zoom_start = max(0, min_len - N_ZOOM)
zoom_end = min_len
zoom_rounds = common_rounds[zoom_start:zoom_end]
x1, x2 = zoom_rounds[0], zoom_rounds[-1]
mid_round = (x1 + x2) // 2

fig = plt.figure(figsize=(15, 7), constrained_layout=True)
gs = GridSpec(2, 2, width_ratios=[3, 1], height_ratios=[1.5, 1.5],
              wspace=0.05, hspace=0.1, figure=fig)
ax = fig.add_subplot(gs[:, 0])

# Main plot
ax.plot(common_rounds, nu_mean, color='C0', label=r'$\hat{\nu}$ (mean)')
if len(nu_arr) >= 2:
    ax.fill_between(common_rounds, nu_mean - nu_std, nu_mean + nu_std,
                    color='C0', alpha=0.2, label=r'$\pm1$ std ($\hat{\nu}$)')
ax.plot(common_rounds, rho_mean, color='C1', label=r'$\hat{\rho}$ (mean)')
if len(rho_arr) >= 2:
    ax.fill_between(common_rounds, rho_mean - rho_std, rho_mean + rho_std,
                    color='C1', alpha=0.2, label=r'$\pm1$ std ($\hat{\rho}$)')
ax.axhline(CFG.nu_true, color='C0', linestyle='--', alpha=0.7, label=r'$\nu^\dagger$')
ax.axhline(CFG.rho_true, color='C1', linestyle='--', alpha=0.7, label=r'$\rho^\dagger$')
ax.set_xlim(0, 150)
ax.xaxis.set_major_locator(MultipleLocator(25))
ax.xaxis.set_minor_locator(MultipleLocator(5))
ax.grid(True, which='major', alpha=0.4)
ax.grid(True, which='minor', alpha=0.2, linestyle=':')
ax.minorticks_on()
ax.set_xlabel('Round', fontsize=18)
ax.tick_params(labelsize=16)

# Legend ordering
handles, labels = ax.get_legend_handles_labels()
if len(nu_arr) >= 2:
    order = [4, 5, 0, 2, 1, 3]   # true values, mean, std
else:
    order = [2, 3, 0, 1]
handles_ordered = [handles[i] for i in order if i < len(handles)]
labels_ordered = [labels[i] for i in order if i < len(labels)]
ax.legend(handles_ordered, labels_ordered, loc='upper center',
          bbox_to_anchor=(0.5, -0.12), fontsize=18, ncol=3, frameon=False)

# Zoom subplots with dynamic tick labels
nu_mean_zoom = nu_mean[zoom_start:zoom_end]
nu_std_zoom  = nu_std[zoom_start:zoom_end]
rho_mean_zoom = rho_mean[zoom_start:zoom_end]
rho_std_zoom  = rho_std[zoom_start:zoom_end]

ax1 = fig.add_subplot(gs[1, 1])   # bottom-right: nu zoom
ax1.fill_between(zoom_rounds, nu_mean_zoom - nu_std_zoom, nu_mean_zoom + nu_std_zoom,
                 color='C0', alpha=0.4, edgecolor='C0', linewidth=0.8)
ax1.plot(zoom_rounds, nu_mean_zoom, color='C0', linewidth=2)
ax1.axhline(CFG.nu_true, color='C0', linestyle='--', linewidth=1.2, alpha=0.7)
ax1.grid(True, which='major', alpha=0.4)
ax1.grid(True, which='minor', alpha=0.2, linestyle=':')
ax1.minorticks_on()
# Dynamic tick labels (first, middle, last of zoom_rounds)
tick_rounds = [zoom_rounds[0], zoom_rounds[len(zoom_rounds)//2], zoom_rounds[-1]]
ax1.set_xticks(tick_rounds)
ax1.set_xticklabels([str(int(r)) for r in tick_rounds])
ax1.tick_params(labelsize=14)
ax1.set_xlabel('Round', fontsize=14)
ax1.set_title(r'$\hat{\nu}$ zoom', fontsize=14)

ax2 = fig.add_subplot(gs[0, 1])   # top-right: rho zoom
ax2.fill_between(zoom_rounds, rho_mean_zoom - rho_std_zoom, rho_mean_zoom + rho_std_zoom,
                 color='C1', alpha=0.4, edgecolor='C1', linewidth=0.8)
ax2.plot(zoom_rounds, rho_mean_zoom, color='C1', linewidth=2)
ax2.axhline(CFG.rho_true, color='C1', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.grid(True, which='major', alpha=0.4)
ax2.grid(True, which='minor', alpha=0.2, linestyle=':')
ax2.minorticks_on()
ax2.set_xticks(tick_rounds)
ax2.set_xticklabels([str(int(r)) for r in tick_rounds])
ax2.tick_params(labelsize=14)
ax2.set_xlabel('Round', fontsize=14)
ax2.set_title(r'$\hat{\rho}$ zoom', fontsize=14)

# Connection lines from main plot to zooms
con1 = ConnectionPatch(xyA=(common_rounds[-1], nu_mean[-1]), xyB=(0, 0.5),
                       coordsA='data', coordsB='axes fraction',
                       axesA=ax, axesB=ax1, color='C0',
                       linestyle='--', linewidth=1.2, alpha=0.6)
con2 = ConnectionPatch(xyA=(common_rounds[-1], rho_mean[-1]), xyB=(0, 0.5),
                       coordsA='data', coordsB='axes fraction',
                       axesA=ax, axesB=ax2, color='C1',
                       linestyle='--', linewidth=1.2, alpha=0.6)
fig.add_artist(con1)
fig.add_artist(con2)

# Save figure
save_dir = ROOT / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(save_dir / "Ex2_HYCO_multi_noise_convergence_zoom.pdf", dpi=600, bbox_inches='tight')
plt.savefig(save_dir / "Ex2_HYCO_multi_noise_convergence_zoom.png", dpi=600, bbox_inches='tight')
print(f"Saved figure to {save_dir / 'HYCO_multi_noise_convergence_zoom.pdf'}")
plt.show()

# ================== 4. Generate LaTeX table ==================
print("\n" + "="*60)
print("Generating HYCO multi-noise LaTeX table with relative L2 errors...")
print("="*60)

# Extract data for table
noise_list = [r["noise"] for r in all_results]
nu_ests = [r["nu"] for r in all_results]
rho_ests = [r["rho"] for r in all_results]
delta_nus = [abs(nu - CFG.nu_true) / CFG.nu_true for nu in nu_ests]
delta_rhos = [abs(rho - CFG.rho_true) / CFG.rho_true for rho in rho_ests]
rel_l2_phy = [r["rel_l2_phy"] for r in all_results]
rel_l2_syn = [r["rel_l2_syn"] for r in all_results]
# Interaction errors (evaluated on a fixed set, not the training set)
# We load them from previous computation, but if not computed, set to nan.
# In this version we keep them as e_int_eval.
# Since we haven't computed them in this block, we can either compute or leave as placeholder.
# We'll compute them using the same ghost_test_pts as before.
fixed_ghost = np.load(ROOT / "data" / "fixed_points.npz")
ghost_test_pts = fixed_ghost["ghost_test_pts"]
e_int_eval = []
for res in all_results:
    nu_est = res["nu"]
    rho_est = res["rho"]
    cube_phys = physical_solver.solve(nu_est, rho_est, store_times)
    phys_vals = sample_cube_polar(cube_phys, physical_solver.r, physical_solver.theta,
                                  store_times, ghost_test_pts)
    # Synthetic prediction
    model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
    model.load_state_dict(torch.load(res["subdir"] / "checkpoints" / "hyco_synthetic.pt", map_location=DEVICE))
    model.eval()
    z_ghost = points_polar_to_network(ghost_test_pts)
    syn_vals = evaluate_network_numpy(model, z_ghost, output_index=1)
    e_int = float(np.mean((phys_vals - syn_vals) ** 2))
    e_int_eval.append(e_int)

runtimes = [r["runtime"] for r in all_results]

def fmt_sci(x):
    return f"{x:.3e}" if x < 0.01 else f"{x:.6f}"
def fmt_float(x, digits=6):
    return f"{x:.{digits}f}"

latex_lines = []
latex_lines.append(r"\begin{table}[htbp]")
latex_lines.append(r"    \centering")
latex_lines.append(r"    \begin{tabular}{@{}lccccc@{}}")
latex_lines.append(r"        \toprule")
latex_lines.append(r"        \multirow{2}{*}{Metric} & \multicolumn{5}{c}{\(\eta\)} \\")
latex_lines.append(r"        \cmidrule(lr){2-6}")
latex_lines.append(r"        & " + " & ".join([f"{nl:.2f}" for nl in noise_list]) + r" \\")
latex_lines.append(r"        \midrule")
rows = [
    (r"$\hat\nu$", [fmt_sci(v) for v in nu_ests]),
    (r"$\hat\rho$", [fmt_float(v, 5) for v in rho_ests]),
    (r"$\delta\nu$", [fmt_float(v, 5) for v in delta_nus]),
    (r"$\delta\rho$", [fmt_float(v, 5) for v in delta_rhos]),
    (r"$e_{L2}^{\rm phy}$", [fmt_sci(v) for v in rel_l2_phy]),      # relative L2 for HYCO-PHY
    (r"$e_{L2}^{\rm syn}$", [fmt_sci(v) for v in rel_l2_syn]),      # relative L2 for HYCO-SYN
    (r"$e_{\rm int}^{\rm eval}$", [fmt_sci(v) for v in e_int_eval]), # renamed evaluation interaction error
    ("Runtime (s)", [f"{v:.2f}" for v in runtimes]),
]
for label, values in rows:
    row = f"        {label} & " + " & ".join(values) + r" \\"
    latex_lines.append(row)
latex_lines.append(r"        \bottomrule")
latex_lines.append(r"    \end{tabular}")
latex_lines.append(r"    \caption{Parameter recovery, relative $L^2$ errors, and ghost-point agreement under increasing noise $\eta$.}")
latex_lines.append(r"    \label{tab:hyco-noise-param}")
latex_lines.append(r"\end{table}")

table_save_dir = ROOT / "tables"
table_save_dir.mkdir(parents=True, exist_ok=True)
table_path = table_save_dir / "HYCO_noise_table.tex"
with open(table_path, "w", encoding="utf-8") as f:
    f.write("\n".join(latex_lines))

print(f"\nLaTeX table saved to {table_path}")
print("\n" + "="*60)
print("LaTeX Table:")
print("="*60)
print("\n".join(latex_lines))
print("="*60)
print("Block 5 completed: Figure and table saved.")

In [ ]:
# ================================
# Block 6 (revised): Comparison Table (HYCO, PINN, XPINN) with separate HYCO-PHY and HYCO-SYN errors
# ================================
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Load HYCO (noise=0.00)
hyco_dir = ROOT / "hyco" / "noise_00"
hyco_hist = pd.read_csv(hyco_dir/"logs"/"hyco_history.csv")
hyco_res = np.load(hyco_dir/"data"/"hyco_results.npz")
nu_hyco = hyco_res["nu"].item()
rho_hyco = hyco_res["rho"].item()
runtime_hyco = hyco_res["runtime"].item()

# Load PINN
pinn_dir = ROOT / "pinn"
try:
    pinn_res = np.load(pinn_dir/"data"/"pinn_results.npz")
    nu_pinn = pinn_res["nu"].item()
    rho_pinn = pinn_res["rho"].item()
    runtime_pinn = pinn_res["runtime"].item()
    pinn_hist = pd.read_csv(pinn_dir/"logs"/"pinn_history.csv")
except FileNotFoundError:
    print("PINN results not found. Set to NaN.")
    nu_pinn = rho_pinn = runtime_pinn = np.nan
    pinn_hist = None

# Load XPINN
xpinn_dir = ROOT / "xpinn"
try:
    xpinn_res = np.load(xpinn_dir/"data"/"xpinn_results.npz")
    nu_xpinn = xpinn_res["nu"].item()
    rho_xpinn = xpinn_res["rho"].item()
    runtime_xpinn = xpinn_res["runtime"].item()
    xpinn_hist = pd.read_csv(xpinn_dir/"logs"/"xpinn_history.csv")
except FileNotFoundError:
    print("XPINN results not found. Set to NaN.")
    nu_xpinn = rho_xpinn = runtime_xpinn = np.nan
    xpinn_hist = None

# Relative errors
def rel_error(est, true):
    if np.isnan(est):
        return np.nan
    return abs(est - true) / true

delta_nu_hyco = rel_error(nu_hyco, CFG.nu_true)
delta_rho_hyco = rel_error(rho_hyco, CFG.rho_true)
delta_nu_pinn = rel_error(nu_pinn, CFG.nu_true)
delta_rho_pinn = rel_error(rho_pinn, CFG.rho_true)
delta_nu_xpinn = rel_error(nu_xpinn, CFG.nu_true)
delta_rho_xpinn = rel_error(rho_xpinn, CFG.rho_true)

# Load fixed points and ground truth
fixed = np.load(ROOT/"data"/"fixed_points.npz")
obs_phy_pts = fixed["obs_phy_pts"]
obs_syn_pts = fixed["obs_syn_pts"]
validation_pts = fixed["validation_pts"]
ghost_test_pts = fixed["ghost_test_pts"]

truth = np.load(ROOT/"data"/"reference_solution.npz")
truth_cube = truth["omega"]
r_grid = truth["r"]
theta_grid = truth["theta"]
times = truth["times"]

val_phy_true = sample_cube_polar(truth_cube, r_grid, theta_grid, times, obs_phy_pts)
val_syn_true = sample_cube_polar(truth_cube, r_grid, theta_grid, times, obs_syn_pts)
val_val_true = sample_cube_polar(truth_cube, r_grid, theta_grid, times, validation_pts)

# Define sector splitting
def split_by_sector(points):
    theta = points[:, 1]
    phys_mask = physical_sector_mask(theta)
    syn_mask = synthetic_sector_mask(theta)
    return points[phys_mask], points[syn_mask]

val_phys, val_syn = split_by_sector(validation_pts)

# Ground truth for subsets
if len(val_phys) > 0:
    val_phys_true_sub = sample_cube_polar(truth_cube, r_grid, theta_grid, times, val_phys)
else:
    val_phys_true_sub = np.array([])
if len(val_syn) > 0:
    val_syn_true_sub = sample_cube_polar(truth_cube, r_grid, theta_grid, times, val_syn)
else:
    val_syn_true_sub = np.array([])

# --- Compute errors for models (neural networks) ---
def compute_errors(model_type, output_index=1):
    if model_type == 'hyco':
        net = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
        net.load_state_dict(torch.load(hyco_dir/"checkpoints"/"hyco_synthetic.pt", map_location=DEVICE))
    elif model_type == 'pinn':
        if not (pinn_dir/"checkpoints"/"pinn.pt").exists():
            return {k: np.nan for k in ['e_phy','e_syn','e_phy_val','e_syn_val']}
        net = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
        ckpt = torch.load(pinn_dir/"checkpoints"/"pinn.pt", map_location=DEVICE)
        net.load_state_dict(ckpt["model"])
    elif model_type == 'xpinn':
        if not (xpinn_dir/"checkpoints"/"xpinn.pt").exists():
            return {k: np.nan for k in ['e_phy','e_syn','e_phy_val','e_syn_val']}
        net = SectorXPINN(max(56, CFG.nn_width//1), CFG.nn_depth, 4).to(DEVICE)
        ckpt = torch.load(xpinn_dir/"checkpoints"/"xpinn.pt", map_location=DEVICE)
        net.load_state_dict(ckpt["model"])
    else:
        raise ValueError("Unknown model_type")
    net.eval()

    pred_phy = evaluate_network_numpy(net, points_polar_to_network(obs_phy_pts), output_index=output_index)
    pred_syn = evaluate_network_numpy(net, points_polar_to_network(obs_syn_pts), output_index=output_index)
    pred_val = evaluate_network_numpy(net, points_polar_to_network(validation_pts), output_index=output_index)

    e_phy = np.mean((pred_phy - val_phy_true)**2)
    e_syn = np.mean((pred_syn - val_syn_true)**2)
    e_val = np.mean((pred_val - val_val_true)**2)

    if len(val_phys) > 0:
        pred_val_phys = evaluate_network_numpy(net, points_polar_to_network(val_phys), output_index=output_index)
        e_phy_val = np.mean((pred_val_phys - val_phys_true_sub)**2)
    else:
        e_phy_val = np.nan
    if len(val_syn) > 0:
        pred_val_syn = evaluate_network_numpy(net, points_polar_to_network(val_syn), output_index=output_index)
        e_syn_val = np.mean((pred_val_syn - val_syn_true_sub)**2)
    else:
        e_syn_val = np.nan

    return {'e_phy': e_phy, 'e_syn': e_syn, 'e_phy_val': e_phy_val, 'e_syn_val': e_syn_val}

# Errors for neural models
hyco_err = compute_errors('hyco', output_index=1)
pinn_err = compute_errors('pinn', output_index=1)
xpinn_err = compute_errors('xpinn', output_index=1)

# --- Additional: HYCO-PHY (physical solver) errors on the two sectors ---
# Re-solve with HYCO parameters
hyco_phy_cube = physical_solver.solve(nu_hyco, rho_hyco, times)
# On physical observations
hyco_phy_pred_phy = sample_cube_polar(hyco_phy_cube, physical_solver.r, physical_solver.theta,
                                      times, obs_phy_pts)
hyco_phy_e_phy = np.mean((hyco_phy_pred_phy - val_phy_true)**2)
# On synthetic observations
hyco_phy_pred_syn = sample_cube_polar(hyco_phy_cube, physical_solver.r, physical_solver.theta,
                                      times, obs_syn_pts)
hyco_phy_e_syn = np.mean((hyco_phy_pred_syn - val_syn_true)**2)
# On validation subsets (if any)
if len(val_phys) > 0:
    hyco_phy_pred_val_phys = sample_cube_polar(hyco_phy_cube, physical_solver.r, physical_solver.theta,
                                               times, val_phys)
    hyco_phy_e_phy_val = np.mean((hyco_phy_pred_val_phys - val_phys_true_sub)**2)
else:
    hyco_phy_e_phy_val = np.nan
if len(val_syn) > 0:
    hyco_phy_pred_val_syn = sample_cube_polar(hyco_phy_cube, physical_solver.r, physical_solver.theta,
                                              times, val_syn)
    hyco_phy_e_syn_val = np.mean((hyco_phy_pred_val_syn - val_syn_true_sub)**2)
else:
    hyco_phy_e_syn_val = np.nan

# --- HYCO-SYN errors are already in hyco_err (the neural network) ---
# But we want separate rows for HYCO-PHY and HYCO-SYN, not the aggregated HYCO (which is same as HYCO-SYN for the network)
# So we will use hyco_err for HYCO-SYN, and the computed hyco_phy_* for HYCO-PHY.

# Assemble data rows
methods = ['HYCO-PHY', 'HYCO-SYN', 'PINN', 'XPINN']
# For HYCO-PHY we don't have nu/rho estimates separately? We use the same nu_hyco/rho_hyco.
nu_ests = [nu_hyco, nu_hyco, nu_pinn, nu_xpinn]
rho_ests = [rho_hyco, rho_hyco, rho_pinn, rho_xpinn]
delta_nu = [rel_error(nu_hyco, CFG.nu_true), rel_error(nu_hyco, CFG.nu_true),
            delta_nu_pinn, delta_nu_xpinn]
delta_rho = [rel_error(rho_hyco, CFG.rho_true), rel_error(rho_hyco, CFG.rho_true),
             delta_rho_pinn, delta_rho_xpinn]

# e_phy(Ω) and e_syn(Ω) for each method
e_phy_Omega = [hyco_phy_e_phy, hyco_err['e_phy'], pinn_err['e_phy'], xpinn_err['e_phy']]
e_syn_Omega = [hyco_phy_e_syn, hyco_err['e_syn'], pinn_err['e_syn'], xpinn_err['e_syn']]
e_phy_Omega1 = [hyco_phy_e_phy_val, hyco_err['e_phy_val'], pinn_err['e_phy_val'], xpinn_err['e_phy_val']]
e_syn_Omega1 = [hyco_phy_e_syn_val, hyco_err['e_syn_val'], pinn_err['e_syn_val'], xpinn_err['e_syn_val']]

# Gaps (only meaningful for HYCO-SYN, PINN, XPINN)
gap_phy = [np.nan, hyco_err['e_phy'] - hyco_err['e_phy_val'],
           pinn_err['e_phy'] - pinn_err['e_phy_val'],
           xpinn_err['e_phy'] - xpinn_err['e_phy_val']]
gap_syn = [np.nan, hyco_err['e_syn'] - hyco_err['e_syn_val'],
           pinn_err['e_syn'] - pinn_err['e_syn_val'],
           xpinn_err['e_syn'] - xpinn_err['e_syn_val']]
runtimes = [runtime_hyco, runtime_hyco, runtime_pinn, runtime_xpinn]  # HYCO-PHY and HYCO-SYN share runtime

def fmt(x, digits=5):
    if np.isnan(x):
        return "--"
    if x < 1e-4:
        return f"{x:.3e}"
    else:
        return f"{x:.7f}"

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{10pt}")
lines.append(r"\caption{Comparison of HYCO (PHY and SYN), PINN, and XPINN on the Navier–Stokes inverse problem (noise level $\eta=0.00$).}")
lines.append(r"\label{tab:hyco_pinn_compare:02}")
lines.append(r"\begin{tabular}{@{\hspace{2mm}} l cccc @{\hspace{2mm}}}")
lines.append(r"\toprule")
lines.append(r"\multirow{2}{*}{Metric} & \multicolumn{4}{c}{Method} \\")
lines.append(r"\cmidrule(lr){2-5}")
lines.append(r"                        & HYCO-PHY & HYCO-SYN & PINN & XPINN \\")
lines.append(r"\midrule")
lines.append(r"$\hat\nu$               & " + " & ".join([fmt(v) for v in nu_ests]) + r" \\")
lines.append(r"$\hat\rho$              & " + " & ".join([fmt(v) for v in rho_ests]) + r" \\")
lines.append(r"$\delta\nu$             & " + " & ".join([fmt(v) for v in delta_nu]) + r" \\")
lines.append(r"$\delta\rho$            & " + " & ".join([fmt(v) for v in delta_rho]) + r" \\")
lines.append(r"$e_{\rm phy}(\Omega)$   & " + " & ".join([fmt(v) for v in e_phy_Omega]) + r" \\")
lines.append(r"$e_{\rm syn}(\Omega)$   & " + " & ".join([fmt(v) for v in e_syn_Omega]) + r" \\")
lines.append(r"$e_{\rm phy}(\Omega_1)$ & " + " & ".join([fmt(v) for v in e_phy_Omega1]) + r" \\")
lines.append(r"$e_{\rm syn}(\Omega_1)$ & " + " & ".join([fmt(v) for v in e_syn_Omega1]) + r" \\")
lines.append(r"$\mathrm{Gap}_{\rm phy}$ & " + " & ".join([fmt(v) for v in gap_phy]) + r" \\")
lines.append(r"$\mathrm{Gap}_{\rm syn}$ & " + " & ".join([fmt(v) for v in gap_syn]) + r" \\")
lines.append(r"\midrule")
lines.append(r"Runtime (s)             & " + " & ".join([f"{v:.1f}" if not np.isnan(v) else "--" for v in runtimes]) + r" \\")
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

save_dir = ROOT / "tables"
save_dir.mkdir(parents=True, exist_ok=True)
table_path = save_dir / "comparison_table.tex"
with open(table_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"Saved comparison table to {table_path}")
print("\n" + "="*60)
print("LaTeX Table:")
print("="*60)
print("\n".join(lines))
print("="*60)

In [ ]:
# ================================
# Block 7 (revised): Snapshots and error evolution with corrected output_index for HYCO-SYN
# ================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Circle
from matplotlib.colors import PowerNorm
import warnings
import torch
from pathlib import Path

warnings.filterwarnings("ignore", message="The input coordinates to pcolormesh are interpreted as cell centers")

mpl.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

root = ROOT
truth = np.load(root / "data" / "reference_solution.npz")
truth_cube = truth["omega"]
store_times = truth["times"]
true_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true,
                                     CFG.dt_true, CFG.final_time)
X, Y = true_solver.X, true_solver.Y
nr, nt = X.shape

target_times = [0.0, 0.5, 1.0, 1.5]
time_indices = [np.argmin(np.abs(store_times - t)) for t in target_times]
selected_times = store_times[time_indices]

# HYCO
hyco_dir = ROOT / "hyco" / "noise_00"
hyco_res = np.load(hyco_dir / "data" / "hyco_results.npz")
nu_hyco = hyco_res["nu"].item()
rho_hyco = hyco_res["rho"].item()
hyco_phy_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true,
                                         CFG.dt_true, CFG.final_time)
hyco_phy_cube = hyco_phy_solver.solve(nu_hyco, rho_hyco, store_times)
hyco_phy_snap = hyco_phy_cube[time_indices]

hyco_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
hyco_model.load_state_dict(torch.load(hyco_dir / "checkpoints" / "hyco_synthetic.pt",
                                      map_location=DEVICE))
hyco_model.eval()

def predict_on_grid(model, t_idx, output_index=1, batch_size=8192):   
    t_val = store_times[t_idx]
    x_flat = X.flatten()
    y_flat = Y.flatten()
    t_norm = np.full_like(x_flat, t_val / CFG.final_time, dtype=np.float32)
    z_np = np.column_stack([x_flat, y_flat, t_norm])
    preds = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(z_np), batch_size):
            batch = to_tensor(z_np[i:i+batch_size])
            out = model(batch)
            if out.shape[1] > 1:
                out = out[:, output_index:output_index+1]
            preds.append(out.cpu().numpy())
    return np.concatenate(preds, axis=0).reshape(nr, nt)

# Now call with output_index=1 explicitly (though default is 1)
hyco_syn_snap = np.array([predict_on_grid(hyco_model, idx, output_index=1)
                          for idx in time_indices])

# PINN
pinn_dir = root / "pinn"
pinn_ckpt = torch.load(pinn_dir / "checkpoints" / "pinn.pt", map_location=DEVICE)
pinn_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
pinn_model.load_state_dict(pinn_ckpt["model"])
pinn_snap = np.array([predict_on_grid(pinn_model, idx, output_index=1)
                      for idx in time_indices])

# XPINN
xpinn_dir = root / "xpinn"
xpinn_ckpt = torch.load(xpinn_dir / "checkpoints" / "xpinn.pt", map_location=DEVICE)
xpinn_model = SectorXPINN(max(56, CFG.nn_width // 1), CFG.nn_depth, 4).to(DEVICE)
xpinn_model.load_state_dict(xpinn_ckpt["model"])
xpinn_snap = np.array([predict_on_grid(xpinn_model, idx, output_index=1)
                       for idx in time_indices])

true_snap = truth_cube[time_indices]

error_hyco_phy = np.abs(hyco_phy_snap - true_snap)
error_hyco_syn = np.abs(hyco_syn_snap - true_snap)
error_pinn     = np.abs(pinn_snap - true_snap)
error_xpinn    = np.abs(xpinn_snap - true_snap)

# Percentile normalization
def get_percentile_limits(data_list, lower=1, upper=99):
    all_vals = np.concatenate([d.flatten() for d in data_list])
    vmin = np.percentile(all_vals, lower)
    vmax = np.percentile(all_vals, upper)
    return vmin, vmax

pred_list = [true_snap, hyco_phy_snap, hyco_syn_snap, pinn_snap, xpinn_snap]
pred_vmin, pred_vmax = get_percentile_limits(pred_list, 1, 99)

err_list = [error_hyco_phy, error_hyco_syn, error_pinn, error_xpinn]
err_vmin = 0.0
err_vmax = np.percentile(np.concatenate([d.flatten() for d in err_list]), 98)
if err_vmax == 0:
    err_vmax = 1.0

print(f"Prediction limits: [{pred_vmin:.3f}, {pred_vmax:.3f}]")
print(f"Error limits: [{err_vmin:.3f}, {err_vmax:.3f}]")

# Plotting function
def plot_snapshots(data_list, row_labels, is_error, save_name,
                   vmin, vmax, cmap, draw_contour=True, cbar_aspect=40):
    n_rows = len(data_list)
    n_cols = 4
    height_ratios = [1.0] * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 2.5 * n_rows),
                             gridspec_kw={'height_ratios': height_ratios,
                                          'wspace': 0.05, 'hspace': 0.05},
                             subplot_kw={'aspect': 'equal'})
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    images = []
    for i_row, (data, label) in enumerate(zip(data_list, row_labels)):
        for j_col in range(n_cols):
            ax = axes[i_row, j_col]
            snapshot = data[j_col] if data is not None else None
            if snapshot is None:
                ax.axis('off')
                continue
            norm = PowerNorm(gamma=0.55, vmin=vmin, vmax=vmax)
            im = ax.pcolormesh(X, Y, snapshot, shading='auto',
                               cmap=cmap, norm=norm)
            ax.set_xlim(-1.1, 1.1)
            ax.set_ylim(-1.1, 1.1)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            edge_color = 'black' if is_error else 'white'
            for r in [CFG.r_inner, CFG.r_outer]:
                circle = Circle((0, 0), r, transform=ax.transData,
                                fill=False, edgecolor=edge_color,
                                linewidth=0.8, linestyle='--', alpha=0.7)
                ax.add_patch(circle)
            if draw_contour:
                levels = np.linspace(vmin, vmax, 12)
                contour_color = 'black' if is_error else 'gray'
                ax.contour(X, Y, snapshot, levels=levels,
                           colors=contour_color, linewidths=0.4, alpha=0.6)
            if i_row == 0:
                ax.set_title(f't={selected_times[j_col]:.1f}', fontsize=12, pad=6)
            if j_col == 0:
                ax.set_ylabel(label, fontsize=10, rotation=90, labelpad=10)
            images.append(im)
    if images:
        im_ref = images[0]
        cbar = fig.colorbar(im_ref, ax=axes, orientation='vertical',
                            fraction=0.015, pad=0.02,
                            aspect=cbar_aspect,
                            location='right')
    save_dir = ROOT / "figures"
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / f"{save_name}.pdf", dpi=600, bbox_inches='tight')
    fig.savefig(save_dir / f"{save_name}.png", dpi=600, bbox_inches='tight')
    print(f"Saved {save_name}.pdf and .png in {save_dir}")
    plt.show()
    return fig

# Plot Figure 1: solution evolution
pred_data = [true_snap, hyco_phy_snap, hyco_syn_snap, pinn_snap, xpinn_snap]
pred_labels = ['TRUE', 'HYCO-PHY', 'HYCO-SYN', 'PINN', 'XPINN']
fig1 = plot_snapshots(
    data_list=pred_data,
    row_labels=pred_labels,
    is_error=False,
    save_name="Ex2_solution_evolution_powernorm",
    vmin=pred_vmin,
    vmax=pred_vmax,
    cmap='RdBu_r',
    draw_contour=True,
    cbar_aspect=82
)

# Plot Figure 2: absolute error evolution
err_data = [error_hyco_phy, error_hyco_syn, error_pinn, error_xpinn]
err_labels = ['HYCO-PHY', 'HYCO-SYN', 'PINN', 'XPINN']
fig2 = plot_snapshots(
    data_list=err_data,
    row_labels=err_labels,
    is_error=True,
    save_name="Ex2_error_evolution_powernorm",
    vmin=err_vmin,
    vmax=err_vmax,
    cmap='RdBu_r',
    draw_contour=True,
    cbar_aspect=66
)

print("Block 7 completed with PowerNorm (gamma=0.55) and corrected HYCO-SYN output index.")

In [ ]:
# ================================
# Block 8: Solution, Zoom, and Error (final time)
# Modification: contour line opacity in error plots set to 0.6 to match Block 7
# ================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
from matplotlib.colors import PowerNorm, Normalize
from matplotlib.cm import ScalarMappable
import torch
from pathlib import Path
from matplotlib.patches import Circle

# High-quality plotting parameters
mpl.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ------------------------------
# 1. Load data and models (load all time instants for error range)
# ------------------------------
root = ROOT
truth = np.load(root / "data" / "reference_solution.npz")
truth_cube = truth["omega"]          # (n_store, nr, nt)
store_times = truth["times"]
true_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true,
                                     CFG.dt_true, CFG.final_time)
X, Y = true_solver.X, true_solver.Y   # (nr, nt)
nr, nt = X.shape

# Final time index
t_final_idx = -1
t_final = store_times[t_final_idx]
print(f"Final time: t = {t_final:.2f}")

# All time indices (4 instants, same as Block 7)
target_times = [0.0, 0.5, 1.0, 1.5]
time_indices = [np.argmin(np.abs(store_times - t)) for t in target_times]
selected_times = store_times[time_indices]

# Ground truth (all instants and final)
true_snap_all = truth_cube[time_indices]   # (4, nr, nt)
true_snap = true_snap_all[-1]               # final instant

# HYCO (noise=0.00)
hyco_dir = ROOT / "hyco" / "noise_00"
hyco_res = np.load(hyco_dir / "data" / "hyco_results.npz")
nu_hyco = hyco_res["nu"].item()
rho_hyco = hyco_res["rho"].item()
hyco_phy_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true,
                                         CFG.dt_true, CFG.final_time)
hyco_phy_cube = hyco_phy_solver.solve(nu_hyco, rho_hyco, store_times)
hyco_phy_all = hyco_phy_cube[time_indices]   # (4, nr, nt)
hyco_phy_snap = hyco_phy_all[-1]             # final instant

# Prediction function (for HYCO-SYN, PINN, XPINN)
def predict_on_grid(model, t_idx, output_index=1, batch_size=8192):
    t_val = store_times[t_idx]
    x_flat = X.flatten()
    y_flat = Y.flatten()
    t_norm = np.full_like(x_flat, t_val / CFG.final_time, dtype=np.float32)
    z_np = np.column_stack([x_flat, y_flat, t_norm])
    preds = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(z_np), batch_size):
            batch = to_tensor(z_np[i:i+batch_size])
            out = model(batch)
            if out.shape[1] > 1:
                out = out[:, output_index:output_index+1]
            preds.append(out.cpu().numpy())
    return np.concatenate(preds, axis=0).reshape(nr, nt)

# HYCO synthetic
hyco_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
hyco_model.load_state_dict(torch.load(hyco_dir / "checkpoints" / "hyco_synthetic.pt",
                                      map_location=DEVICE))
hyco_model.eval()
hyco_syn_all = np.array([predict_on_grid(hyco_model, idx, output_index=1)
                         for idx in time_indices])
hyco_syn_snap = hyco_syn_all[-1]

# PINN
pinn_dir = root / "pinn"
pinn_ckpt = torch.load(pinn_dir / "checkpoints" / "pinn.pt", map_location=DEVICE)
pinn_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
pinn_model.load_state_dict(pinn_ckpt["model"])
pinn_all = np.array([predict_on_grid(pinn_model, idx, output_index=1)
                     for idx in time_indices])
pinn_snap = pinn_all[-1]

# XPINN
xpinn_dir = root / "xpinn"
xpinn_ckpt = torch.load(xpinn_dir / "checkpoints" / "xpinn.pt", map_location=DEVICE)
xpinn_model = SectorXPINN(max(56, CFG.nn_width // 1), CFG.nn_depth, 4).to(DEVICE)
xpinn_model.load_state_dict(xpinn_ckpt["model"])
xpinn_all = np.array([predict_on_grid(xpinn_model, idx, output_index=1)
                      for idx in time_indices])
xpinn_snap = xpinn_all[-1]

# Organize final-time data (for plotting)
method_order = ["TRUE", "Bi-HYCO-PHY", "Bi-HYCO-SYN", "PINN", "XPINN"]
sol_data = {
    "TRUE": true_snap,
    "Bi-HYCO-PHY": hyco_phy_snap,
    "Bi-HYCO-SYN": hyco_syn_snap,
    "PINN": pinn_snap,
    "XPINN": xpinn_snap,
}
err_data = {
    "TRUE": np.zeros_like(true_snap),
    "Bi-HYCO-PHY": np.abs(hyco_phy_snap - true_snap),
    "Bi-HYCO-SYN": np.abs(hyco_syn_snap - true_snap),
    "PINN": np.abs(pinn_snap - true_snap),
    "XPINN": np.abs(xpinn_snap - true_snap),
}

# Compute error colorbar range (using errors from all instants, consistent with Block 7)
err_all_phy = np.abs(hyco_phy_all - true_snap_all)
err_all_syn = np.abs(hyco_syn_all - true_snap_all)
err_all_pinn = np.abs(pinn_all - true_snap_all)
err_all_xpinn = np.abs(xpinn_all - true_snap_all)

all_err_global = np.concatenate([
    err_all_phy.ravel(),
    err_all_syn.ravel(),
    err_all_pinn.ravel(),
    err_all_xpinn.ravel()
])
vmax_err = np.percentile(all_err_global, 98) if np.max(all_err_global) > 0 else 1.0
vmin_err = 0.0
print(f"Block 8 Global Error vmax (all times): {vmax_err:.6f}")

# ------------------------------
# Color scales: solution uses PowerNorm, error also uses PowerNorm
# ------------------------------
pred_list_all_times = [true_snap_all, hyco_phy_all, hyco_syn_all, pinn_all, xpinn_all]
all_sol_all_times = np.concatenate([d.ravel() for d in pred_list_all_times])
vmin_sol = np.percentile(all_sol_all_times, 1)
vmax_sol = np.percentile(all_sol_all_times, 99)
norm_sol = PowerNorm(gamma=0.55, vmin=vmin_sol, vmax=vmax_sol)

# Error uses PowerNorm(gamma=0.55) to match Block 7 exactly
norm_err = PowerNorm(gamma=0.55, vmin=vmin_err, vmax=vmax_err)

# Contour levels
line_levels_sol = np.linspace(vmin_sol, vmax_sol, 12)
line_levels_err = np.linspace(vmin_err, vmax_err, 12)

# Zoom region
ZOOM_XLIM = (-0.35, 0.35)
ZOOM_YLIM = (-0.35, 0.35)

# ------------------------------
# Plotting helper functions (modified plot_err contour alpha to 0.6)
# ------------------------------
def overlay_boundaries(ax, edgecolor='black', linewidth=1.0, linestyle='--', alpha=0.6):
    for r in [CFG.r_inner, CFG.r_outer]:
        circle = Circle((0,0), r, transform=ax.transData,
                        fill=False, edgecolor=edgecolor,
                        linewidth=linewidth, linestyle=linestyle, alpha=alpha)
        ax.add_patch(circle)

def plot_global(ax, data, title, norm, cmap='RdBu_r'):
    im = ax.pcolormesh(X, Y, data, shading='auto', cmap=cmap, norm=norm)
    ax.contour(X, Y, data, levels=line_levels_sol, colors='gray', linewidths=0.32, alpha=0.6)
    overlay_boundaries(ax, edgecolor='white')
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=16)
    ax.set_xticks([]); ax.set_yticks([])
    # Explicitly turn off spines (in case axis('off') doesn't fully take effect)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    return im

def plot_zoom(ax, data, norm, cmap='RdBu_r'):
    im = ax.pcolormesh(X, Y, data, shading='auto', cmap=cmap, norm=norm)
    ax.contour(X, Y, data, levels=line_levels_sol, colors='gray', linewidths=0.8, alpha=0.6)
    overlay_boundaries(ax, edgecolor='black')
    ax.set_xlim(ZOOM_XLIM); ax.set_ylim(ZOOM_YLIM)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    # Explicitly turn off spines (in case axis('off') doesn't fully take effect)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return im

def plot_err(ax, data, norm, cmap='RdBu_r'):
    im = ax.pcolormesh(X, Y, data, shading='auto', cmap=cmap, norm=norm)
    # <--- Modification: contour line opacity set to 0.6 to match Block 7 exactly --->
    ax.contour(X, Y, data, levels=line_levels_err, colors='k', linewidths=0.4, alpha=0.6)
    overlay_boundaries(ax, edgecolor='black')
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    # Explicitly turn off spines (in case axis('off') doesn't fully take effect)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    return im

# ------------------------------
# Create figure
# ------------------------------
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(3, 5, wspace=0.05, hspace=0.06,
                       height_ratios=[1, 0.8, 1])

ax_sol_glob = [fig.add_subplot(gs[0, j]) for j in range(5)]
ax_sol_zoom = [fig.add_subplot(gs[1, j]) for j in range(5)]
ax_err = [fig.add_subplot(gs[2, j]) for j in range(5)]

cf_sol_glob = []
for j, name in enumerate(method_order):
    cf = plot_global(ax_sol_glob[j], sol_data[name], name, norm_sol)
    cf_sol_glob.append(cf)

for j, name in enumerate(method_order):
    plot_zoom(ax_sol_zoom[j], sol_data[name], norm_sol)

for j, name in enumerate(method_order):
    if name == "TRUE":
        ax_err[j].axis('off')
    else:
        plot_err(ax_err[j], err_data[name], norm_err)

# Color bars
fig.subplots_adjust(right=0.88, left=0.06, bottom=0.08, top=0.94)
cbar_ax_sol = fig.add_axes([0.90, 0.45, 0.02, 0.48])
cbar_sol = fig.colorbar(cf_sol_glob[0], cax=cbar_ax_sol)
cbar_sol.ax.tick_params(labelsize=14)

sm_err = ScalarMappable(norm=norm_err, cmap='RdBu_r')
sm_err.set_array([])
cbar_ax_err = fig.add_axes([0.90, 0.08, 0.02, 0.28])
cbar_err = fig.colorbar(sm_err, cax=cbar_ax_err)
cbar_err.ax.tick_params(labelsize=14)

# Row labels
row_labels = ["Solution", "Solution Zoom", "Error"]
y_positions = [0.86, 0.59, 0.32]
for label, y in zip(row_labels, y_positions):
    fig.text(0.02, y, label, rotation='vertical',
             va='center', ha='center', fontsize=20)

# Save
save_dir = root / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
save_name = "Ex2_solution_zoom_error_final_time_powernorm_err"
fig.savefig(save_dir / (save_name + ".pdf"), dpi=600, bbox_inches='tight')
fig.savefig(save_dir / (save_name + ".png"), dpi=600, bbox_inches='tight')
print(f"Figure saved to {save_dir / (save_name + '.pdf')}")
plt.show()

In [ ]:
# ================================
# Block 9: Gradient Magnitude (3D, 2D Global, 2D Zoom) - WITH CONTOUR ON ZOOM
# ================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
from matplotlib.colors import PowerNorm
from matplotlib.cm import ScalarMappable
from mpl_toolkits.mplot3d import Axes3D
import torch
from pathlib import Path
from matplotlib.patches import Circle

mpl.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ------------------------------
# 1. Load data (reuse fields from Block 8)
# ------------------------------
root = ROOT
truth = np.load(root / "data" / "reference_solution.npz")
truth_cube = truth["omega"]
store_times = truth["times"]
true_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true,
                                     CFG.dt_true, CFG.final_time)
X, Y = true_solver.X, true_solver.Y
nr, nt = X.shape

t_final_idx = -1
true_snap = truth_cube[t_final_idx]

# HYCO
hyco_dir = ROOT / "hyco" / "noise_00"
hyco_res = np.load(hyco_dir / "data" / "hyco_results.npz")
nu_hyco = hyco_res["nu"].item()
rho_hyco = hyco_res["rho"].item()
hyco_phy_solver = AnnularVorticitySolver(CFG.nr_true, CFG.ntheta_true,
                                         CFG.dt_true, CFG.final_time)
hyco_phy_snap = hyco_phy_solver.solve(nu_hyco, rho_hyco, store_times)[t_final_idx]

hyco_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
hyco_model.load_state_dict(torch.load(hyco_dir / "checkpoints" / "hyco_synthetic.pt",
                                      map_location=DEVICE))
hyco_model.eval()

def predict_final(model, output_index=1):
    t_val = store_times[t_final_idx]
    x_flat = X.flatten()
    y_flat = Y.flatten()
    t_norm = np.full_like(x_flat, t_val / CFG.final_time, dtype=np.float32)
    z_np = np.column_stack([x_flat, y_flat, t_norm])
    preds = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(z_np), CFG.eval_batch):
            batch = to_tensor(z_np[i:i+CFG.eval_batch])
            out = model(batch)
            if out.shape[1] > 1:
                out = out[:, output_index:output_index+1]
            preds.append(out.cpu().numpy())
    return np.concatenate(preds, axis=0).reshape(nr, nt)

hyco_syn_snap = predict_final(hyco_model, output_index=1)

# PINN
pinn_dir = root / "pinn"
pinn_ckpt = torch.load(pinn_dir / "checkpoints" / "pinn.pt", map_location=DEVICE)
pinn_model = MixedPINN(CFG.nn_width, CFG.nn_depth).to(DEVICE)
pinn_model.load_state_dict(pinn_ckpt["model"])
pinn_snap = predict_final(pinn_model, output_index=1)

# XPINN
xpinn_dir = root / "xpinn"
xpinn_ckpt = torch.load(xpinn_dir / "checkpoints" / "xpinn.pt", map_location=DEVICE)
xpinn_model = SectorXPINN(max(56, CFG.nn_width // 1), CFG.nn_depth, 4).to(DEVICE)
xpinn_model.load_state_dict(xpinn_ckpt["model"])
xpinn_snap = predict_final(xpinn_model, output_index=1)

method_order = ["TRUE", "Bi-HYCO-PHY", "Bi-HYCO-SYN", "PINN", "XPINN"]
sol_data = {
    "TRUE": true_snap,
    "Bi-HYCO-PHY": hyco_phy_snap,
    "Bi-HYCO-SYN": hyco_syn_snap,
    "PINN": pinn_snap,
    "XPINN": xpinn_snap,
}

# ------------------------------
# 2. Accurate computation of gradient magnitude (polar coordinates)
# ------------------------------
def gradient_magnitude_polar(U):
    """Compute gradient magnitude on polar grid |∇ω| = sqrt( (ω_r)^2 + (ω_θ/r)^2 )"""
    nr, nt = U.shape
    r = true_solver.r
    dtheta = true_solver.dtheta
    dr = true_solver.dr
    
    # Radial derivative
    omega_r = np.zeros_like(U)
    omega_r[1:-1, :] = (U[2:, :] - U[:-2, :]) / (2 * dr)
    omega_r[0, :] = (U[1, :] - U[0, :]) / dr
    omega_r[-1, :] = (U[-1, :] - U[-2, :]) / dr
    
    # Angular derivative (periodic boundary)
    omega_theta = np.zeros_like(U)
    omega_theta[:, 1:-1] = (U[:, 2:] - U[:, :-2]) / (2 * dtheta)
    omega_theta[:, 0] = (U[:, 1] - U[:, -1]) / (2 * dtheta)
    omega_theta[:, -1] = (U[:, 0] - U[:, -2]) / (2 * dtheta)
    
    # Divide by r
    R = r[:, None]
    R = np.maximum(R, 1e-12)
    grad_r = omega_r
    grad_theta = omega_theta / R
    
    return np.sqrt(grad_r**2 + grad_theta**2)

# Compute all gradients
grad_data = {}
for name in method_order:
    grad_data[name] = gradient_magnitude_polar(sol_data[name])

# Check if data is valid
for name in method_order:
    data = grad_data[name]
    print(f"Gradient {name}: min={data.min():.3e}, max={data.max():.3e}, mean={data.mean():.3e}")
    if np.all(data == 0):
        print(f"WARNING: Gradient for {name} is all zeros!")

# ------------------------------
# 3. Color scale
# ------------------------------
all_grad = np.concatenate([grad_data[name].ravel() for name in method_order])
all_grad = all_grad[np.isfinite(all_grad)]
if all_grad.size == 0:
    raise ValueError("All gradient data are NaN or inf. Check gradient computation.")
vmin_grad = np.percentile(all_grad, 1)
vmax_grad = np.percentile(all_grad, 99)
if vmin_grad == vmax_grad:
    vmin_grad -= 0.5
    vmax_grad += 0.5
norm_grad = PowerNorm(gamma=0.55, vmin=vmin_grad, vmax=vmax_grad)

# Contour levels (for Zoom plot)
contour_levels = np.linspace(vmin_grad, vmax_grad, 12)

# Zoom region
ZOOM_XLIM = (-0.35, 0.35)
ZOOM_YLIM = (-0.35, 0.35)

# ------------------------------
# 4. Plotting helpers (Zoom plot adds contours)
# ------------------------------
def overlay_boundaries(ax, edgecolor='black'):
    for r in [CFG.r_inner, CFG.r_outer]:
        circle = Circle((0,0), r, transform=ax.transData,
                        fill=False, edgecolor=edgecolor,
                        linewidth=1.0, linestyle='--', alpha=0.6)
        ax.add_patch(circle)

def plot_grad_3d(ax, data, title, norm, cmap='RdBu_r'):
    surf = ax.plot_surface(X, Y, data, cmap=cmap, norm=norm,
                           edgecolor='none', alpha=0.9, rstride=1, cstride=1)
    th = np.linspace(0, 2*np.pi, 200)
    for r in [CFG.r_inner, CFG.r_outer]:
        xb = r * np.cos(th)
        yb = r * np.sin(th)
        ax.plot(xb, yb, 0, color='black', lw=1.0, alpha=0.85)
    ax.set_box_aspect((1, 1, 0.5), zoom=1.4)
    ax.set_title(title, fontsize=16)
    ax.axis('off')
    ax.view_init(elev=35, azim=-45)
    return surf

def plot_grad_2d(ax, data, norm, cmap='RdBu_r'):
    im = ax.pcolormesh(X, Y, data, shading='auto', cmap=cmap, norm=norm)
    overlay_boundaries(ax)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    # Explicitly turn off spines (in case axis('off') doesn't fully take effect)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    return im

def plot_grad_2d_zoom(ax, data, norm, cmap='RdBu_r'):
    # Draw filled contour
    im = ax.pcolormesh(X, Y, data, shading='auto', cmap=cmap, norm=norm)
    # Add contour lines (black dashed)
    ax.contour(X, Y, data, levels=contour_levels, colors='k', linewidths=0.5, alpha=0.6, linestyles='--')
    overlay_boundaries(ax)
    ax.set_xlim(ZOOM_XLIM); ax.set_ylim(ZOOM_YLIM)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    # Explicitly turn off spines (in case axis('off') doesn't fully take effect)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return im

# ------------------------------
# 5. Create figure
# ------------------------------
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(3, 5, wspace=0.05, hspace=0.06,
                       height_ratios=[1, 1, 0.8])

ax_3d = [fig.add_subplot(gs[0, j], projection='3d') for j in range(5)]
ax_2d = [fig.add_subplot(gs[1, j]) for j in range(5)]
ax_2d_zoom = [fig.add_subplot(gs[2, j]) for j in range(5)]

surf_list = []
for j, name in enumerate(method_order):
    data = grad_data[name]
    surf = plot_grad_3d(ax_3d[j], data, name, norm_grad)
    surf_list.append(surf)
    plot_grad_2d(ax_2d[j], data, norm_grad)
    plot_grad_2d_zoom(ax_2d_zoom[j], data, norm_grad)

# Color bar
fig.subplots_adjust(right=0.88, left=0.06, bottom=0.08, top=0.94)
cbar_ax = fig.add_axes([0.90, 0.08, 0.02, 0.85])
cbar = fig.colorbar(surf_list[0], cax=cbar_ax)
cbar.ax.tick_params(labelsize=14)

# Row labels
row_labels = ["Gradient (3D)", "Gradient (2D)", "Gradient (2D Zoom)"]
y_positions = [0.85, 0.53, 0.22]
for label, y in zip(row_labels, y_positions):
    fig.text(0.02, y, label, rotation='vertical',
             va='center', ha='center', fontsize=20)

# Save
save_dir = root / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
save_name = "Ex2_gradient_3d_2d_zoom_final_time_contour"
fig.savefig(save_dir / (save_name + ".pdf"), dpi=600, bbox_inches='tight')
fig.savefig(save_dir / (save_name + ".png"), dpi=600, bbox_inches='tight')
print(f"Figure saved to {save_dir / (save_name + '.pdf')}")
plt.show()